# Description 

This notebook shows the results interpreter component. 

As before, the component comprises:

- A configuration 

- DomainData

- DomainTool 

+ a run function that takes a user query and returns either a str or a TaskResult 


# Initial target questions 

- query2_1 = "why 12 wells werent modelled from the initial screened ones?"
- query2_2 = "which is the best injector? in terms of utility"
- query2_3 = "whats the best supported producer?"
- query2_4 = "sumarize the quality of the models"
- query2_5 = "Is there any evidence of thieve zones?"
- query2_6 = "Plot the observed/simulated liquid rates for all the wells. " #analyst will take care of this. 
- query2_7 = "is there any evidence of aquifer support?"                    # fail 
- query2_8 = "Did we use BHP in the simulation?"                            # in the simulation?"                             
- query2_9 = "rank the injectors based on their utility"
- query2_10 = """for all producer wells, pick 3-4 points in their production history and for thos points plot the " \
observed liquid produced and simulated
"""
- query2_11 ="Give me a summary of the results, focus on support and chanelling"
- query2_12 = "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?"




# Dependencies


## External dependencies

In [1]:
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')


import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict,  List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if


imported


## Internal dependencies 


In [2]:
from agentic_system.common.base_domain_tools import BaseDataComponent, BaseDomainTools
from agentic_system.common.get_llm import azure_llm_if as get_llm 


imported


# Initialize 

#### Mock data (taken from real cases)

- Model Logs
- Processing Logs  
- CRM-Liquid history match parameters table 
- CRM-Liquid history match rates table 
- Configuration file for the run
- Maybe injectors table 

for simplicity, we will use the same names as the files exported by the simulator 

Examples: 

- processing_logs.json, 

- historical_liquid_rates.csv 

- historical_liquid_crm.csv 

#### Initial target questions 

- query2_1 = "why 12 wells werent modelled from the initial screened ones?"
- query2_2 = "which is the best injector? in terms of utility"
- query2_3 = "whats the best supported producer?"
- query2_4 = "sumarize the quality of the models"
- query2_5 = "Is there any evidence of thieve zones?"
- query2_6 = "Plot the observed/simulated liquid rates for all the wells. "
- query2_7 = "is there any evidence of aquifer support?"                # fail 
- query2_8 = "Did we use BHP in the simulation?                                                    
- query2_9 = "rank the injectors based on their utility"
- query2_10 = """for all producer wells, pick 3-4 points in their production history and for thos points plot the observed liquid produced and simulated """
- query2_11 ="Give me a summary of the results, focus on support and chanelling"
- query2_12 = "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?"


# Onthologies  ? and subsurface responses ?  

CRM simulation background and objectives 

Injector utility 

Producer utility 

Producer support 

Stranded wells 

Connectivity 

Deficit and Aquifer support

Characteristic response time

High/Low injection rate, High/Low production 



? Reallocation
? Redistribution




##### We might need some pre-processing before making the data available 

In [3]:
import pandas as pd 
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')



In [4]:

DATAFOLDER = "../../datasets/CRMResultsExample"
study_name = "ForAgents"
liquid_history_file = f"{DATAFOLDER}/historical_liquid_crm.csv"

liquid_crm = pd.read_csv( liquid_history_file )
if "ALLOCATION" in liquid_crm:
    liquid_crm.rename({"ALLOCATION":"GAIN"},axis=1,inplace=True)
liquid_crm['PALLOCATION'] = 0.75 *  liquid_crm['GAIN']
liquid_crm

,INJECTOR,PRODUCER,GAIN,TAU,TAUP,PRODUCTIVITY,LO,MODEL,ID,R2,BIAS_RATIO,CORRELATION,VARIANCE_RATIO,QUALITY_SCORE,SUBZONE,PALLOCATION
0,BG-1639_I,BG-2016_P,0.003472,10.293859,34.411707,0.0,1.194710,OneLayerCRMPSingleConstrained,0,-2.477959,0.961771,0.137599,1.659276,0.384530,ALLWARA,0.002604
1,BG-1799_I,BG-2016_P,0.049873,10.293859,34.411707,0.0,1.194710,OneLayerCRMPSingleConstrained,1,-2.477959,0.961771,0.137599,1.659276,0.384530,ALLWARA,0.037405
2,BG-1801_I,BG-2016_P,0.111089,10.293859,34.411707,0.0,1.194710,OneLayerCRMPSingleConstrained,2,-2.477959,0.961771,0.137599,1.659276,0.384530,ALLWARA,0.083317
3,BG-1957_I,BG-2016_P,0.000006,10.293859,34.411707,0.0,1.194710,OneLayerCRMPSingleConstrained,3,-2.477959,0.961771,0.137599,1.659276,0.384530,ALLWARA,0.000005
4,BG-1639_I,BG-2031_P,0.003376,48.684777,1.000000,0.0,1.000000,OneLayerCRMPSingleConstrained,4,0.293903,0.208798,0.997122,0.207998,0.413118,ALLWARA,0.002532
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1286,BG-2044_I,BG-1922_P,0.000029,0.500000,23.087441,0.0,0.048096,OneLayerCRMPSingleConstrained,1286,0.386908,0.999895,0.622034,0.617813,0.719121,ALLWARA,0.000022
1287,BG-2053_I,BG-1922_P,0.000091,0.500000,23.087441,0.0,0.048096,OneLayerCRMPSingleConstrained,1287,0.386908,0.999895,0.622034,0.617813,0.719121,ALLWARA,0.000068
1288,BG-2047_I,BG-2046_P,0.997503,0.525601,1.000000,0.0,1.000000,OneLayerCRMPSingleConstrained,1288,-4.850028,0.227819,0.466023,0.508528,0.351323,ALLWARA,0.748127
1289,BG-0740_I,BG-0760_P,0.000894,6.772576,23.732836,0.0,0.982152,OneLayerCRMPSingleConstrained,1289,0.932911,1.000937,0.965896,0.959762,0.973814,ALLWARA,0.000671


In [5]:
import pandas as pd

class CRMResultsPreprocess:

    NEGLIGIBLE_GAIN = 0.05
    VERY_LOW_GAIN = 0.10

    QUALITY_VERY_GOOD = 0.8
    QUALITY_GOOD = 0.6
    QUALITY_POOR = 0.4

    BHP_PRODUCTIVITY_TOLERANCE = 0.0001


    @classmethod
    def classify_gain(cls, gain: float) -> str:
        """
        Classify injector-producer connectivity strength.
        """

        if gain < cls.NEGLIGIBLE_GAIN:
            return "negligible"

        if gain < cls.VERY_LOW_GAIN:
            return "very_low"

        return "meaningful"


    @classmethod
    def classify_quality(cls, score: float) -> str:
        """
        Classify producer model quality from QUALITY_SCORE.
        """

        if score > cls.QUALITY_VERY_GOOD:
            return "very_good"

        if score >= cls.QUALITY_GOOD:
            return "good"

        if score >= cls.QUALITY_POOR:
            return "poor"

        return "very_poor"



    @classmethod
    def build_injector_summary(
        cls,
        connectivity: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Build one row per injector summarizing modeled connectivity.

        Injector utility is currently defined as:

            utility = sum(GAIN)

        Negligible connections (GAIN < NEGLIGIBLE_GAIN) contribute to the
        utility sum but do NOT count toward the number of supported producers.

        Returns
        -------
        pd.DataFrame
            Columns:
            - injector
            - utility
            - n_supported_producers
            - max_gain
            - dominant_producer
            - dominant_producer_quality_score
            - dominant_producer_quality_class
            - subzone, when available
        """

        required_columns = {
            "injector",
            "producer",
            "gain",
            "producer_quality_score",
            "producer_quality_class",
        }

        missing = required_columns - set(connectivity.columns)

        if missing:
            raise ValueError(
                f"Missing required connectivity columns: {sorted(missing)}"
            )

        rows = []

        for injector, group in connectivity.groupby("injector", sort=False):

            # Utility includes all fitted gains, including negligible ones.
            utility = group["gain"].sum()

            meaningful = group[
                group["gain"] >= cls.NEGLIGIBLE_GAIN
            ]

            n_supported_producers = meaningful["producer"].nunique()

            # Pair with the strongest modeled connectivity.
            dominant_idx = group["gain"].idxmax()
            dominant = group.loc[dominant_idx]

            row = {
                "injector": injector,
                "utility": utility,
                "n_supported_producers": n_supported_producers,
                "max_gain": dominant["gain"],
                "dominant_producer": dominant["producer"],
                "dominant_producer_quality_score":
                    dominant["producer_quality_score"],
                "dominant_producer_quality_class":
                    dominant["producer_quality_class"],
            }

            if "subzone" in group.columns:
                subzones = group["subzone"].dropna().unique()

                row["subzone"] = (
                    subzones[0]
                    if len(subzones) == 1
                    else ",".join(map(str, subzones))
                )

            rows.append(row)

        df = pd.DataFrame(rows)

        return (
            df
            .sort_values("utility", ascending=False)
            .reset_index(drop=True)
        )


    @classmethod
    def build_producer_support_summary(
        cls,
        connectivity: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Build one row per producer summarizing modeled injector support.

        Producer support is currently defined as:

            support = sum(GAIN)

        Negligible connections (GAIN < NEGLIGIBLE_GAIN) contribute to the
        support sum but do NOT count toward the number of supporting injectors.

        Returns
        -------
        pd.DataFrame
            Columns:
            - producer
            - support
            - n_supporting_injectors
            - max_gain
            - dominant_injector
            - quality_score
            - quality_class
            - subzone, when available
        """

        required_columns = {
            "injector",
            "producer",
            "gain",
            "producer_quality_score",
            "producer_quality_class",
        }

        missing = required_columns - set(connectivity.columns)

        if missing:
            raise ValueError(
                f"Missing required connectivity columns: {sorted(missing)}"
            )

        rows = []

        for producer, group in connectivity.groupby("producer", sort=False):

            # Support includes all fitted gains.
            support = group["gain"].sum()

            meaningful = group[
                group["gain"] >= cls.NEGLIGIBLE_GAIN
            ]

            n_supporting_injectors = meaningful["injector"].nunique()

            dominant_idx = group["gain"].idxmax()
            dominant = group.loc[dominant_idx]

            # Quality is producer-level and therefore should be constant
            # across every injector row for this producer.
            quality_scores = group["producer_quality_score"].unique()
            quality_classes = group["producer_quality_class"].unique()

            if len(quality_scores) != 1 or len(quality_classes) != 1:
                raise ValueError(
                    f"Inconsistent producer quality values for producer {producer}"
                )

            row = {
                "producer": producer,
                "support": support,
                "n_supporting_injectors": n_supporting_injectors,
                "max_gain": dominant["gain"],
                "dominant_injector": dominant["injector"],
                "quality_score": quality_scores[0],
                "quality_class": quality_classes[0],
            }

            if "subzone" in group.columns:
                subzones = group["subzone"].dropna().unique()

                row["subzone"] = (
                    subzones[0]
                    if len(subzones) == 1
                    else ",".join(map(str, subzones))
                )

            rows.append(row)

        df = pd.DataFrame(rows)

        return (
            df
            .sort_values("support", ascending=False)
            .reset_index(drop=True)
        )

    @classmethod
    def build_connectivity_table(
        cls,
        liquid_crm: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Build a normalized injector-producer connectivity table.

        Returns one row per modeled injector-producer pair.

        Semantic fields:
        - gain:
            Fraction of injector injection associated with liquid response
            at the producer.
        - pallocation:
            Fraction of the producer liquid production attributed to this injector.
        - gain_class:
            Connectivity strength classification based on GAIN.
        - producer_quality_score:
            History-match quality of the producer model.
        """

        required_columns = {
            "INJECTOR",
            "PRODUCER",
            "GAIN",
            "PALLOCATION",
            "QUALITY_SCORE",
        }

        missing = required_columns - set(liquid_crm.columns)

        if missing:
            raise ValueError(
                f"Missing required columns in liquid CRM results: {sorted(missing)}"
            )

        columns = [
            "INJECTOR",
            "PRODUCER",
            "GAIN",
            "PALLOCATION",
            "QUALITY_SCORE",
        ]

        if "SUBZONE" in liquid_crm.columns:
            columns.append("SUBZONE")

        df = liquid_crm[columns].copy()

        df = df.rename(
            columns={
                "INJECTOR": "injector",
                "PRODUCER": "producer",
                "GAIN": "gain",
                "PALLOCATION": "pallocation",
                "QUALITY_SCORE": "producer_quality_score",
                "SUBZONE": "subzone",
            }
        )

        numeric_columns = [
            "gain",
            "pallocation",
            "producer_quality_score",
        ]

        for column in numeric_columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

        if df[numeric_columns].isna().any().any():

            bad_columns = (
                df[numeric_columns]
                .columns[df[numeric_columns].isna().any()]
                .tolist()
            )

            raise ValueError(
                "Invalid or missing numeric values found in connectivity "
                f"columns: {bad_columns}"
            )

        df["gain_class"] = df["gain"].apply(
            cls.classify_gain
        )

        df["producer_quality_class"] = (
            df["producer_quality_score"]
            .apply(cls.classify_quality)
        )

        return df.reset_index(drop=True)


    @classmethod
    def build_producer_liquid_model_table(
        cls,
        liquid_crm: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Build a producer-level CRM liquid model table.

        Producer-level parameters and quality metrics are repeated in the raw
        table once per injector-producer pair. This method reduces them to one
        row per producer.

        bhp_used is simulation-level information. If any fitted producer has a
        non-negligible productivity value, BHP is considered to have been used
        by the simulation and the same value is assigned to every producer row.
        """

        required_columns = {
            "PRODUCER",
            "TAU",
            "TAUP",
            "PRODUCTIVITY",
            "LO",
            "BIAS_RATIO",
            "CORRELATION",
            "VARIANCE_RATIO",
            "QUALITY_SCORE",
        }

        missing = required_columns - set(liquid_crm.columns)

        if missing:
            raise ValueError(
                f"Missing required columns in liquid CRM results: {sorted(missing)}"
            )

        columns = [
            "PRODUCER",
            "TAU",
            "TAUP",
            "PRODUCTIVITY",
            "LO",
            "BIAS_RATIO",
            "CORRELATION",
            "VARIANCE_RATIO",
            "QUALITY_SCORE",
        ]

        if "SUBZONE" in liquid_crm.columns:
            columns.append("SUBZONE")

        df = liquid_crm[columns].copy()

        value_columns = [
            column
            for column in columns
            if column != "PRODUCER"
        ]

        inconsistent = (
            df.groupby("PRODUCER")[value_columns]
            .nunique(dropna=False)
            .gt(1)
            .any(axis=1)
        )

        bad_producers = (
            inconsistent[inconsistent]
            .index
            .tolist()
        )

        if bad_producers:
            raise ValueError(
                "Producer-level CRM values are not constant for producers: "
                f"{bad_producers}"
            )

        df = (
            df
            .drop_duplicates(subset=["PRODUCER"])
            .copy()
        )

        df = df.rename(
            columns={
                "PRODUCER": "producer",
                "TAU": "tau",
                "TAUP": "taup",
                "PRODUCTIVITY": "productivity",
                "LO": "lo",
                "BIAS_RATIO": "bias_ratio",
                "CORRELATION": "correlation",
                "VARIANCE_RATIO": "variance_ratio",
                "QUALITY_SCORE": "quality_score",
                "SUBZONE": "subzone",
            }
        )

        numeric_columns = [
            "tau",
            "taup",
            "productivity",
            "lo",
            "bias_ratio",
            "correlation",
            "variance_ratio",
            "quality_score",
        ]

        for column in numeric_columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

        if df[numeric_columns].isna().any().any():

            bad_columns = (
                df[numeric_columns]
                .columns[df[numeric_columns].isna().any()]
                .tolist()
            )

            raise ValueError(
                "Invalid or missing numeric values found in producer model "
                f"columns: {bad_columns}"
            )

        # BHP usage is simulation-level, not producer-level.
        bhp_used = bool(
            (df["productivity"] > cls.BHP_PRODUCTIVITY_TOLERANCE).any()
        )

        df["bhp_used"] = bhp_used

        df["quality_class"] = (
            df["quality_score"]
            .apply(cls.classify_quality)
        )

        return df.reset_index(drop=True)

    @classmethod
    def process_liquid_crm(
        cls,
        liquid_crm: pd.DataFrame,
    ) -> dict[str, pd.DataFrame]:

        connectivity = cls.build_connectivity_table(
            liquid_crm
        )

        producer_liquid_model = cls.build_producer_liquid_model_table(
            liquid_crm
        )

        injector_summary = cls.build_injector_summary(
            connectivity
        )

        producer_support_summary = cls.build_producer_support_summary(
            connectivity
        )

        return {
            "crm_connectivity": connectivity,
            "producer_liquid_model": producer_liquid_model,
            "injector_summary": injector_summary,
            "producer_support_summary": producer_support_summary,
        }

tables = CRMResultsPreprocess.process_liquid_crm( liquid_crm )
print(tables.keys())


tables['crm_connectivity'].head(3)

dict_keys(['crm_connectivity', 'producer_liquid_model', 'injector_summary', 'producer_support_summary'])


,injector,producer,gain,pallocation,producer_quality_score,subzone,gain_class,producer_quality_class
0,BG-1639_I,BG-2016_P,0.003472,0.002604,0.38453,ALLWARA,negligible,very_poor
1,BG-1799_I,BG-2016_P,0.049873,0.037405,0.38453,ALLWARA,negligible,very_poor
2,BG-1801_I,BG-2016_P,0.111089,0.083317,0.38453,ALLWARA,meaningful,very_poor


# First prototype 

In [7]:

import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
from typing_extensions import Self
from typing import Any, cast
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from agentic_system.common.base_domain_tools import BaseDataComponent, BaseDomainTools

class ResultsInterpreterData( BaseDataComponent ):

    #def __init__(self):
    #    super().__init__()
           
    
    def fetch_data_item_and_context( self, dataset_name: str )->str: 
        return 'dummy' 

    def record_response(self, question:str, response:str )->str:
        return 'dummy' 
    
    def interpret_data_item( self, question, dataset_name )->str:
        return  'dummy' 

    def _get_simulation_config( self ):
        """
        Retrieves the simulation configuration file as a JSON string and a description of 
        the parameters defined there. This file contains information about all the parameters used in the simulation.
        """
        pass 

    def _get_simulation_logs( self ):
        """
        Retrieves the simulation logs. These contain information on:
        - Errors in the modelling of specific wells
        - Reasons why specific wells were not modelled 
        - Other logs produced by the simulation engine 
        """
        pass 


    #def get_injector_performance_metrics():
    #    pass 

    #def get_injector_performance_metrics():
    #       pass 
       

    def used_bhp(self)->bool:
        return True 
    
class ResultsInterpreterTools(  BaseDomainTools[ResultsInterpreterData] ):

    #def __init__(self):
    #    super().__init__()


    def generate_summary(self):
        """Generates a summary of simulation results"""
        return "all results were produced, status = 200"



    @property
    def data(self) -> ResultsInterpreterData:
        return self.data_component

    @property
    def raw_data(self) -> Any:
        return self.data_component.raw_data 


    def compute_injector_summary(self)->pd.DataFrame:
        """
        Computes a summary of injector performance based on the CRM results.
        """
        raw_data = self.raw_data 

        return pd.DataFrame()  # Placeholder implementation

class ResultsInterpreterConfig( ):
    pass 

class ResultsInterpreterComponent( ):

    def __init__(self, llm, config:ResultsInterpreterConfig, data:ResultsInterpreterData, tools:ResultsInterpreterTools ):
        self.llm = llm 
        self.config = config 
        self.data_component = data 
        self.domain_tools = tools 
        self.domain_tools.set_data_component( self.data_component )

    def set_data(self, data:Any, metadata:Any|None = None)->Self:
        self.data_component.set_data(data, metadata)
        return self
    
    def run( self, query:str, background:str|None = None )->Any:
        message =  "[run] I am a hard-coded runner for CRMResultsAnalyst that creates an agent inplace"
        print(message)

        agent_tools = self.domain_tools.get_agent_tools()
        prompt = "You help with questions related to CRM results"

        agent = create_agent(model = self.llm,
                             system_prompt= prompt,
                             tools=agent_tools
                             )

      
        response = agent.invoke({"messages": [{"role": "user", "content": query}]})


        return response  

        # need to generate a basic plan 
    


In [ ]:
import io
import pandas as pd

c = """
INJECTOR	PRODUCER PALLOCATION	GAIN	GAIN_CLASS  
0	I1	P1	0.508456	0.38   meaningful
1	I2	P1	0.432001	0.30   meaningful
2	I3	P1	0.007279	0.001  negligible
3	I1	P2	0.463442	0.4    meaningful
4	I3	P2	0.226982	0.15   meaningful
"""
q="""	PRODUCER	CORRELATION	VARIANCE_RATIO	QUALITY_SCORE	QUALITY_CLASS
0	P1	0.802377	0.929430	0.864767	good
1	P2	0.929300	1.324738	0.886701	good
2	P3	0.754035	1.020565	0.833949	good
3	P4	0.925126	1.205418	0.917859	good
"""
p="""
  PRODUCER  current_produced_water_fraction  current_produced_oil_fraction  current_volume_liquid_produced  current_liquid_production_due_to_depletion  current_liquid_production_due_to_injection  current_liquid_production_due_to_pressure  pressure_coefficient  total_allocation  number_supporting_injectors
0   P1                             0.72                           0.28                          1200.0                               420.0                              660.0                              120.0                  0.18              0.55                           2
1   P2                             0.35                           0.65                           950.0                               570.0                              285.0                               95.0                  0.10              0.30                           2
2   P3                             0.88                           0.12                           700.0                               140.0                              490.0                               70.0                  0.22              0.70                           1
3   P4                             0.20                           0.80                          1500.0                              1050.0                              300.0                              150.0                  0.08              0.20                           0
"""


# Read the string into a DataFrame
# 'sep=r"\s+"' handles any whitespace (tabs or multiple spaces)
# 'index_col=0' uses the first column (0, 1, 2...) as the row index
connectivity_table  = pd.read_csv(io.StringIO(c), sep=r"\s+", index_col=0)
simulation_quality_table = pd.read_csv(io.StringIO(q), sep=r"\s+", index_col=0)
producer_model_table = pd.read_csv(io.StringIO(p), sep=r"\s+", index_col=0)


print(connectivity_table)
print(simulation_quality_table)
print(producer_model_table.T)


raw_data = {
    "connectivity_table": connectivity_table,
    "simulation_quality_table": simulation_quality_table,
    "producer_model_table": producer_model_table,
}



  INJECTOR PRODUCER  PALLOCATION   GAIN  GAIN_CLASS
0       I1       P1     0.508456  0.380  meaningful
1       I2       P1     0.432001  0.300  meaningful
2       I3       P1     0.007279  0.001  negligible
3       I1       P2     0.463442  0.400  meaningful
4       I3       P2     0.226982  0.150  meaningful
  PRODUCER  CORRELATION  VARIANCE_RATIO  QUALITY_SCORE QUALITY_CLASS
0       P1     0.802377        0.929430       0.864767          good
1       P2     0.929300        1.324738       0.886701          good
2       P3     0.754035        1.020565       0.833949          good
3       P4     0.925126        1.205418       0.917859          good
                                                 0      1      2       3
PRODUCER                                        P1     P2     P3      P4
current_produced_water_fraction               0.72   0.35   0.88     0.2
current_produced_oil_fraction                 0.28   0.65   0.12     0.8
current_volume_liquid_produced              1200.0 

In [11]:
from agentic_system.common.semantic_models import ColumnCard, SemanticCatalog, SemanticContext, TableCard


raw_data = {
    "connectivity_table": connectivity_table,
    "simulation_quality_table": simulation_quality_table,
    "producer_model_table": producer_model_table,
}


semantic_catalog = SemanticCatalog(

    tables=[

        TableCard(
            name="connectivity_table",
            description=(
                "CRM injector-producer connectivity results. "
                "Each row represents one modeled injector-producer pair."
            ),
            kind="derived",
            row_count=len(connectivity_table),
            columns=[
                ColumnCard(
                    name="INJECTOR",
                    data_type="string",
                    description="Injector well name (source)."
                ),
                ColumnCard(
                    name="PRODUCER",
                    data_type="string",
                    description="Producer well name (sink)."
                ),
                ColumnCard(
                    name="GAIN",
                    data_type="float",
                    description="Fraction of water injected in the injector that is recovered as liquid production in the producer. GAIN is denoted as G_{ij}. This is the -gain- of production due to injection. This controls the strength of support from injector (i) to producer (p)"
                ),
                ColumnCard(
                    name="PALLOCATION",
                    data_type="float",
                    description=(
                        "This is the -Producer allocation-. It is the fraction of the producer's total liquid production "
                        "attributed to injection from this injector. The sum of the producer allocation for each producer must be less than 1. The "
                        "total liquid production of a producer includes contributions from all injectors connected to the producer, the producer "
                        "production due to pressure changes and the liquid production due to primary depletion "
                    )
                ),
                ColumnCard(
                    name="GAIN_CLASS",
                    data_type="string",
                    description="Categorical interpretation of GAIN. GAIN < 0.05 is negligible. GAIN in [0.05-0.15] is low. GAIN > 0.7 is high. For a given"
                    

                ),
            ],
        ),

        TableCard(
            name="simulation_quality_table",
            description=(
                "Producer-level CRM history-match quality metrics. "
                "Each row represents one producer model."
            ),
            kind="derived",
            row_count=len(simulation_quality_table),
            columns=[
                ColumnCard(
                    name="PRODUCER",
                    data_type="string",
                    description="Producer well name."
                ),
                ColumnCard(
                    name="CORRELATION",
                    data_type="float",
                    description=(
                        "Correlation between observed and simulated liquid "
                        "production time series."
                    )
                ),
                ColumnCard(
                    name="VARIANCE_RATIO",
                    data_type="float",
                    description=(
                        "Ratio between simulated and observed production variance."
                    )
                ),
                ColumnCard(
                    name="QUALITY_SCORE",
                    data_type="float",
                    description=(
                        "Composite CRM history-match quality metric from 0 to 1. "
                        "Higher values indicate better model quality."
                    )
                ),
                ColumnCard(
                    name="QUALITY_CLASS",
                    data_type="string",
                    description="Categorical interpretation of model quality."
                ),
            ],
        ),

        TableCard(
            name="producer_model_table",
            description=(
                "Current producer-level production state and modeled "
                "production-support decomposition."
            ),
            kind="derived",
            row_count=len(producer_model_table),
            columns=[
                ColumnCard(
                    name="PROCER",
                    data_type="string",
                    description="Producer well name."
                ),
                ColumnCard(
                    name="current_produced_water_fraction",
                    data_type="float",
                    description="Current fraction of produced liquid that is water. Also named -water cut-"
                ),
                ColumnCard(
                    name="current_produced_oil_fraction",
                    data_type="float",
                    description="Current fraction of produced liquid that is oil."
                ),
                ColumnCard(
                    name="current_volume_liquid_produced",
                    data_type="float",
                    description="Latest observed liquid production."
                ),
                ColumnCard(
                    name="current_liquid_production_due_to_depletion",
                    data_type="float",
                    description="Recent liquid production attributed to depletion."
                ),
                ColumnCard(
                    name="current_liquid_production_due_to_injection",
                    data_type="float",
                    description="Recent liquid production attributed to injection."
                ),
                ColumnCard(
                    name="current_liquid_production_due_to_pressure",
                    data_type="float",
                    description="Recent liquid production attributed to pressure changes."
                ),
                ColumnCard(
                    name="pressure_coefficient",
                    data_type="float",
                    description=(
                        "CRM-P productivity coefficient: derivative of liquid "
                        "production with respect to bottom-hole pressure."
                    )
                ),
                ColumnCard(
                    name="total_allocation",
                    data_type="float",
                    description=(
                        "Sum of PALLOCATION over all injectors connected "
                        "to this producer."
                    )
                ),
                ColumnCard(
                    name="number_supporting_injectors",
                    data_type="integer",
                    description=(
                        "Number of injectors having non-negligible GAIN "
                        "to this producer."
                    )
                ),
            ],
        ),
    ],

    semantic_constraints=[
        "GAIN and PALLOCATION have different physical meanings and must not be used interchangeably.",
        "GAIN is injector-centric.",
        "PALLOCATION is producer-centric.",
        "GAIN below 0.05 is considered negligible.",
        "For a balanced simulation, the sum of GAIN across producers for an injector must be <= 1.",
        "The sum of PALLOCATION across injectors for a producer must be <= 1.",
        "Produced water fraction and produced oil fraction must sum to 1.",
    ],
)


semantic_context = SemanticContext(

    definitions=[
        (
            "A producer is supported when one or more injectors have "
            "non-negligible GAIN to that producer."
        ),
        (
            "An injector supports producers when it has non-negligible "
            "GAIN to one or more producers."
        ),
        (
            "Injector utility is related to the sum of GAIN across the "
            "producers supported by that injector."
        ),
    ],

    business_rules=[
        "GAIN < 0.05 is negligible.",
        (
            "A high-utility injector supports one or more producers and "
            "has total GAIN across producers approaching 1 in a balanced model."
        ),
        (
            "Model quality should be considered when assessing confidence "
            "in producer-level interpretations."
        ),
    ],

    domain_knowledge=[
        (
            "A producer may be supported by injection while only a small "
            "fraction of its total liquid production is due to injection."
        ),
        (
            "Total producer liquid production may contain contributions "
            "from injection, depletion, and pressure changes."
        ),
        (
            "total_allocation measures the fraction of producer liquid "
            "production attributable to all modeled injectors combined."
        ),
    ],
)


metadata = {
    "semantic_catalog": semantic_catalog,
    "semantic_context": semantic_context,
}


data_component = ResultsInterpreterData()
data_component.set_data(raw_data, metadata)

In [12]:
prompt_template1 = r"""
You are a Reservoir Engineer specialized in waterflood surveillance, history matching, and Capacitance Resistance Models (CRM).

Your job is to aid in the interpretation of results while answering user questions related to simulation results.

# Simulation Context

The history match uses a CRMP model to reproduce each producer’s liquid-rate history as the sum of injection support, pressure support, and primary production:

\[
q_p(t)
=
\sum_i G_{{ip}}\,R_{{\tau_p}}\left[I_i(t)\right]
+
J_p\,\tau_p\,R_{{\tau_p}}\left[\Delta P_p(t)\right]
+
L_{{o,p}}\,q_{{o,p}}\exp\left(-\frac{{t}}{{\tau_{{p,p}}}}\right)
\]

where:

- **GAIN** (\(G_{{ip}}\)) controls the strength of support from injector \(i\) to producer \(p\).

- **TAU** controls the response delay and smoothing of both injection and pressure effects.

- **PRODUCTIVITY** (\(J_p\)) scales the pressure contribution.

- **Lo** controls the initial magnitude of primary production.

- **TAUP** controls how quickly primary production declines.

- \(R_{{\tau}}[\cdot]\) denotes the model’s exponential response to the corresponding time series.

The parameters are fitted by minimizing the mismatch between observed and simulated liquid production over active production periods.

The parameters found by the simulator are the GAIN, TAU, TAUP, PALOCATION, PRODUCTIVITY, Lo.

Note that the simulation mode can be "balanced" or "unbalanced".

**Balanced:** enforces physical constraints that guarantee

\[
\sum_p G_{{ip}} < 1
\]

for all injectors \(i\). Namely, the sum of GAIN per injector is less than or equal to 1.

**Unbalanced:** physical constraints are relaxed, which allows

\[
\sum_p G_{{ip}} > 1.
\]

Injectors are balanced when

\[
\sum_p G_{{ip}} < 1
\]

for each \(i\).

You must use the results provided, your own knowledge, and the context provided to analyze the results.

You should be able to produce a concise report summarizing simulation results and go into the details when asked to.

# Simulation Data

The simulation results are grouped into a number of tables. You must ground your responses on the information contained in these tables.

You are provided with tools to retrieve these tables from the data.

{tables}

# Interpretation


#
"""



In [17]:
prompt_template = r"""
You are a Reservoir Engineer specialized in waterflood surveillance, history matching, and Capacitance Resistance Models (CRM).

Your role is to interpret CRM-P simulation results and answer questions about well connectivity, injection support, producer response, model quality, pressure support, and primary production.

# Simulation Context

The history match uses a CRM-P model to reproduce each producer’s liquid-rate history as the sum of injection support, pressure support, and primary production:

\[
q_p(t)
=
\sum_i G_{{ip}}\,R_{{\tau_p}}\left[I_i(t)\right]
+
J_p\,\tau_p\,R_{{\tau_p}}\left[\Delta P_p(t)\right]
+
L_{{o,p}}\,q_{{o,p}}\exp\left(-\frac{{t}}{{\tau_{{p,p}}}}\right)
\]

where:

- **GAIN** (\(G_{{ip}}\)) controls the strength of the modeled support from injector \(i\) to producer \(p\).

- **TAU** controls the delay and smoothing of the modeled injection and pressure responses.

- **PRODUCTIVITY** (\(J_p\)) scales the pressure contribution.

- **Lo** controls the initial magnitude of primary production.

- **TAUP** controls how quickly primary production declines.

- \(R_{{\tau}}[\cdot]\) denotes the model’s exponential response to the corresponding time series.

The parameters are fitted by minimizing the mismatch between observed and simulated liquid production over active production periods.

The simulator provides fitted parameters such as GAIN, TAU, TAUP, PRODUCTIVITY, and Lo. It may also provide derived metrics such as PALLOCATION.

# Balancing

The simulation mode can be **balanced** or **unbalanced**.

In balanced mode, the following constraint is enforced for every injector \(i\):

\[
\sum_p G_{{ip}} \leq 1
\]

This means that the total GAIN assigned from an injector to its connected producers cannot exceed 1.

In unbalanced mode, this constraint is not enforced. Therefore,

\[
\sum_p G_{{ip}} > 1
\]

is permitted but is not necessarily present.

Do not assume that an unbalanced simulation has injectors with total GAIN greater than 1. Verify this from the data.

# Simulation Data

The simulation results are organized into tables. You must ground conclusions about this simulation in the information contained in those tables.

You have tools for inspecting table schemas and retrieving or analyzing table contents.

Available tables:

{tables}

# Analysis Procedure

Before answering:

1. Identify the question’s required concepts and metrics.
2. Select only the tables relevant to those concepts.
3. Inspect the relevant table schemas before constructing the analysis.
4. Use the available tools to retrieve, filter, aggregate, join, rank, or compare the data.
5. Check that the available data is sufficient to support the conclusion.
6. Base numerical statements on tool results, not mental calculation.
7. State clearly when a conclusion cannot be established from the available data.

Do not retrieve every table unless the question requires them.


# Interpretation Rules

Use the following GAIN thresholds unless the user specifies different thresholds:

- **GAIN < 0.05:** negligible modeled connectivity.
- **0.05 ≤ GAIN < 0.10:** very low modeled connectivity.
- **GAIN ≥ 0.10:** meaningful modeled connectivity.

Negligible connections must not be counted as meaningful support.

Interpret GAIN as modeled dynamic connectivity or support. A high GAIN does not, by itself, prove:

- incremental oil recovery;
- attributable oil production;
- injection efficiency;
- direct physical communication;
- channeling or the presence of a thief zone.

These conclusions require additional supporting metrics or evidence.

Distinguish between:

- **GAIN:** the modeled fraction or strength of injector support assigned to a producer.
- **PALLOCATION:** the producer-side allocation or contribution metric provided by the simulation.
- **Injector utility:** a derived assessment that may combine meaningful connectivity, supported producers, injection volume, liquid or oil response, and efficiency.
- **Producer support:** the number and strength of meaningful injector connections associated with a producer.

Do not treat these concepts as interchangeable.

A low TAU indicates a relatively fast modeled response. A high TAU indicates a slower and more strongly smoothed response. TAU must be interpreted together with GAIN, model quality, operating history, and available spatial or reservoir context.

Interpret PRODUCTIVITY and pressure support only when the simulation used valid pressure data and the corresponding pressure contribution is available. A zero PRODUCTIVITY value may indicate no modeled pressure contribution, but it does not independently prove that pressure data were unavailable.

Interpret TAUP and Lo as primary-production parameters. Do not infer aquifer support solely from weak fitted primary decline or unexplained production.

# Model Quality and Confidence

When QUALITY_SCORE is available, use these categories:

- **Very good:** QUALITY_SCORE > 0.80
- **Good:** 0.60 < QUALITY_SCORE ≤ 0.80
- **Poor:** 0.40 < QUALITY_SCORE ≤ 0.60
- **Very poor:** QUALITY_SCORE ≤ 0.40

Use model quality to qualify the confidence of all connectivity and support interpretations.

Strong GAIN values from poor-quality models must be treated cautiously.

Do not claim evidence of channeling, thief zones, early water breakthrough, water coning, or aquifer support from a single parameter. Such interpretations must be based on multiple consistent indicators and should be described as potential evidence unless directly demonstrated by the data.

# Ambiguous Questions

If the user asks for the “best” injector, producer, model, or connection, determine the intended ranking criterion.

For example, “best injector” could mean:

- highest total meaningful GAIN;
- largest number of meaningfully supported producers;
- greatest allocated production;
- greatest attributable oil;
- highest injection efficiency;
- best-quality supported connections.

If the intended criterion is clear from the question or context, use it and state the definition applied. If it is materially ambiguous, ask a concise clarification question.

Do not equate total GAIN with attributable oil unless attributable-oil calculations are explicitly available.

# Grounding and Uncertainty

Separate the following in your answer:

- facts directly reported or calculated from the tables;
- engineering interpretations supported by those facts;
- hypotheses that require additional data or validation.

Do not invent values, columns, tables, well names, relationships, or operating conditions.

If required information is unavailable:

- identify what is missing;
- explain why the question cannot be answered reliably;
- state which table, column, or calculation would be required.

When tables disagree, report the inconsistency rather than silently choosing one result.

# Response Style

Answer the user’s question directly.

For concise questions:

1. Give the conclusion first.
2. State the metric or definition used.
3. Provide the most important supporting values.
4. Add a brief qualification if model quality or data limitations affect confidence.

For summary requests, cover:

- overall model quality;
- principal injector-producer connections;
- strongest and weakest producer support;
- injector utility;
- pressure and primary-production contributions, when available;
- potential anomalous behavior;
- important limitations and recommended follow-up checks.

Keep the initial response concise. Provide detailed tables, rankings, calculations, and engineering interpretation when requested.
"""


def build_prompt_tables_context(semantic_catalog):
    context = "" 
    blocks = [] 
    for table in semantic_catalog.tables:

        s = f"*{table.name}*\n{table.description}\nColumns:\n" 
        for c in table.columns:
            s = s + f"- {c.name}: {c.description}\n"
        
        s = s  + "\n" 

        blocks.append( s )

    context = "\n".join( blocks )
    return context 

#context = build_prompt_tables_context(semantic_catalog)
#my_prompt = prompt_template.format( tables=context )
#print( my_prompt )


In [ ]:
class ResultsInterpreterTools(BaseDomainTools[ResultsInterpreterData]):

    def _no_generate_summary(self):
        """Generates a summary of simulation results."""
        return "all results were produced, status = 200"

    @property
    def data(self) -> ResultsInterpreterData:
        return self.data_component

    @property
    def raw_data(self) -> Any:
        return self.data_component.raw_data

    def get_connectivity_table(
        self,
        producer_names: list[str] | None = None,
        injector_names: list[str] | None = None,
    ) -> pd.DataFrame:
        """
        Return CRM injector-producer connectivity results.

        Each row represents one modeled injector-producer pair.

        Important column semantics:
        - GAIN:
          Fraction of water injected in an injector that is recovered as
          liquid production in the producer. GAIN is injector-centric.
        - PALLOCATION:
          Fraction of the producer's total liquid production attributed to
          injection from that specific injector. PALLOCATION is producer-centric.
        - GAIN_CLASS:
          Classification of GAIN. Gains below 0.05 are negligible.

        Parameters
        ----------
        producer_names:
            Optional list of producer well names to keep.

            If None:
                Do not filter by producer. Producers of any name may be returned.

            If provided:
                Return only rows whose PRODUCER value is one of the names
                in this list.

            Example:
                producer_names=["P1", "P2"]
                returns connectivity rows only for producers P1 and P2.

        injector_names:
            Optional list of injector well names to keep.

            If None:
                Do not filter by injector. Injectors of any name may be returned.

            If provided:
                Return only rows whose INJECTOR value is one of the names
                in this list.

            Example:
                injector_names=["I1"]
                returns connectivity rows only for injector I1.

        Filtering behavior
        ------------------
        If both producer_names and injector_names are provided, BOTH filters
        are applied. A row is returned only if its producer is in
        producer_names AND its injector is in injector_names.

        These parameters only filter rows by well name. They do not rank wells,
        apply GAIN thresholds, or change the meaning of the CRM results.
        """

        df = self.raw_data["connectivity_table"]

        if producer_names is not None:
            df = df[df["PRODUCER"].isin(producer_names)]

        if injector_names is not None:
            df = df[df["INJECTOR"].isin(injector_names)]

        return df.copy()

    def get_simulation_quality_table(
        self,
        producer_names: list[str] | None = None,
    ) -> pd.DataFrame:
        """
        Return producer-level CRM history-match quality information.

        Each row represents the simulation quality for one producer.

        Parameters
        ----------
        producer_names:
            Optional list of producer well names to keep.

            If None:
                Return quality information for all producers.

            If provided:
                Return only rows whose PRODUCER value is one of the names
                in this list.

            Example:
                producer_names=["P1", "P3"]
                returns model-quality information only for P1 and P3.

        This parameter only filters producers by name. It does not filter by
        quality score, quality class, correlation, or any other quality metric.
        """

        df = self.raw_data["simulation_quality_table"]

        if producer_names is not None:
            df = df[df["PRODUCER"].isin(producer_names)]

        return df.copy()

    def get_producer_model_table(
        self,
        producer_names: list[str] | None = None,
    ) -> pd.DataFrame:
        """
        Return producer-level current production and modeled-support information.

        Each row represents one producer.

        The table contains information such as:
        - current produced water and oil fractions
        - latest observed liquid production
        - current liquid production attributed to depletion
        - current liquid production attributed to injection
        - current liquid production attributed to pressure changes
        - pressure coefficient
        - total producer allocation from injectors
        - number of supporting injectors

        Parameters
        ----------
        producer_names:
            Optional list of producer well names to keep.

            If None:
                Return producer-model information for all producers.

            If provided:
                Return only rows whose NAME value is one of the producer names
                in this list.

            Example:
                producer_names=["P2"]
                returns the producer-model row for P2 only.

        This parameter only filters rows by producer name. It does not filter
        based on support level, production volume, water fraction, allocation,
        or any other metric.

        Use current_volume_liquid_produced together with current_produced_oil_fraction
        to estimate current oil production for a producer.

        """

        df = self.raw_data["producer_model_table"]

        if producer_names is not None:
            df = df[df["NAME"].isin(producer_names)]

        return df.copy()

    def compute_injector_summary(
        self,
        injector_names: list[str] | None = None,
    ) -> pd.DataFrame:
        """
        Use this tool for injector utility/support questions.

        Do NOT use this tool alone to estimate impact on oil production, because utility
        is based on GAIN and does not account for producer liquid rate or oil fraction.


        Injector utility is currently defined as the sum of GAIN across all
        modeled producer connections for that injector.

        Gains below 0.05 are considered negligible and do not count toward
        the number of supported producers.

        Parameters
        ----------
        injector_names:
            Optional list of injector well names to include in the summary.

            If None:
                Compute the summary for all injectors.

            If provided:
                Compute the summary only for injectors whose names appear in
                this list.

            Example:
                injector_names=["I1", "I3"]
                returns summary rows only for I1 and I3.

        This parameter selects which injectors are summarized. It does not
        specify producers, ranking order, GAIN thresholds, or a top-N limit.

        Returns
        -------
        pd.DataFrame
            One row per injector, including:
            - INJECTOR
            - UTILITY: sum of GAIN across producers
            - NUMBER_SUPPORTED_PRODUCERS: count of producers with GAIN >= 0.05
            - MAX_GAIN: largest individual GAIN for that injector
            - STRONGEST_CONNECTED_PRODUCER: producer associated with MAX_GAIN
        """

        df = self.raw_data["connectivity_table"]

        if injector_names is not None:
            df = df[df["INJECTOR"].isin(injector_names)]

        if df.empty:
            return pd.DataFrame(
                columns=[
                    "INJECTOR",
                    "UTILITY",
                    "NUMBER_SUPPORTED_PRODUCERS",
                    "MAX_GAIN",
                    "STRONGEST_CONNECTED_PRODUCER",
                ]
            )

        summary_rows = []

        for injector, group in df.groupby("INJECTOR", sort=False):

            strongest_idx = group["GAIN"].idxmax()
            strongest_row = group.loc[strongest_idx]

            summary_rows.append(
                {
                    "INJECTOR": injector,
                    "UTILITY": group["GAIN"].sum(),
                    "NUMBER_SUPPORTED_PRODUCERS": (
                        group.loc[group["GAIN"] >= 0.05, "PRODUCER"].nunique()
                    ),
                    "MAX_GAIN": strongest_row["GAIN"],
                    "STRONGEST_CONNECTED_PRODUCER": strongest_row["PRODUCER"],
                }
            )

        return (
            pd.DataFrame(summary_rows)
            .sort_values("UTILITY", ascending=False)
            .reset_index(drop=True)
        )

    

In [18]:
# 1. Initialize data and bind tools


data_component = ResultsInterpreterData()
data_component.set_data(raw_data, metadata)

tools = ResultsInterpreterTools()

tools.set_data_component(data_component)

## Tests 

In [27]:


assert tools.data is data_component
assert tools.raw_data is data_component.raw_data
# 2. Test tools directly, without an LLM

# Full connectivity table
connectivity = tools.get_connectivity_table()

print(connectivity)

# Filter by producer

p1_connectivity = tools.get_connectivity_table(
    producer_names=["P1"]
)

print(p1_connectivity)
assert set(p1_connectivity["PRODUCER"].unique()) == {"P1"}

# Filter by injector

i1_connectivity = tools.get_connectivity_table(
    injector_names=["I1"]
)

print(i1_connectivity)
assert set(i1_connectivity["INJECTOR"].unique()) == {"I1"}

# Both filters

pair_subset = tools.get_connectivity_table(
    producer_names=["P1"],
    injector_names=["I1"],
)

print(pair_subset)

assert set(pair_subset["PRODUCER"].unique()) == {"P1"}
assert set(pair_subset["INJECTOR"].unique()) == {"I1"}

# Simulation quality

p1_quality = tools.get_simulation_quality_table(
    producer_names=["P1"]
)
print(p1_quality)
assert set(p1_quality["PRODUCER"].unique()) == {"P1"}


# Producer model

p1_model = tools.get_producer_model_table(
    producer_names=["P1"]
)
print(p1_model)
assert set(p1_model["NAME"].unique()) == {"P1"}



  INJECTOR PRODUCER  PALLOCATION   GAIN  GAIN_CLASS
0       I1       P1     0.508456  0.380  meaningful
1       I2       P1     0.432001  0.300  meaningful
2       I3       P1     0.007279  0.001  negligible
3       I1       P2     0.463442  0.400  meaningful
4       I3       P2     0.226982  0.150  meaningful
  INJECTOR PRODUCER  PALLOCATION   GAIN  GAIN_CLASS
0       I1       P1     0.508456  0.380  meaningful
1       I2       P1     0.432001  0.300  meaningful
2       I3       P1     0.007279  0.001  negligible
  INJECTOR PRODUCER  PALLOCATION  GAIN  GAIN_CLASS
0       I1       P1     0.508456  0.38  meaningful
3       I1       P2     0.463442  0.40  meaningful
  INJECTOR PRODUCER  PALLOCATION  GAIN  GAIN_CLASS
0       I1       P1     0.508456  0.38  meaningful
  PRODUCER  CORRELATION  VARIANCE_RATIO  QUALITY_SCORE QUALITY_CLASS
0       P1     0.802377         0.92943       0.864767          good
  NAME  current_produced_water_fraction  current_produced_oil_fraction  \
0   P1       

In [28]:
# Injector utility summary

injector_summary = tools.compute_injector_summary()

print(injector_summary)

i1 = injector_summary.loc[
    injector_summary["INJECTOR"] == "I1"
].iloc[0]

assert abs(i1["UTILITY"] - 0.78) < 1e-6
assert i1["NUMBER_SUPPORTED_PRODUCERS"] == 2
assert abs(i1["MAX_GAIN"] - 0.40) < 1e-6
assert i1["STRONGEST_CONNECTED_PRODUCER"] == "P2"

i3 = injector_summary.loc[
    injector_summary["INJECTOR"] == "I3"
].iloc[0]

assert i3["NUMBER_SUPPORTED_PRODUCERS"] == 1



  INJECTOR  UTILITY  NUMBER_SUPPORTED_PRODUCERS  MAX_GAIN  \
0       I1    0.780                           2      0.40   
1       I2    0.300                           1      0.30   
2       I3    0.151                           1      0.15   

  STRONGEST_CONNECTED_PRODUCER  
0                           P2  
1                           P1  
2                           P2  


## First spin

In [19]:
from langchain.agents import create_agent


def build_prompt(data_component: ResultsInterpreterData) -> str:

    metadata = data_component.metadata

    semantic_catalog: SemanticCatalog = metadata["semantic_catalog"]
    semantic_context: SemanticContext = metadata["semantic_context"]

    parts = [
        """
You are a Results Interpreter specialized in CRM simulation results.

Use the available tools to answer questions about:
- injector utility
- producer support
- injector-producer connectivity
- producer production contributions
- simulation quality

Do not assume that one tool is sufficient.

When the user asks about the effect of injectors on producer oil production,
combine injector-producer connectivity information with producer-level current
production information.

Use PALLOCATION to estimate the fraction of producer liquid attributable to a
specific injector.

For questions about oil impact, combine PALLOCATION with the producer's current
liquid production and current oil fraction.

Use injector utility based on GAIN only when the question is specifically about
injector support/utility, not oil production impact.


Use the supplied semantic definitions as authoritative.
Do not invent alternative meanings for domain-specific metrics.
""".strip()
    ]

    if semantic_context.definitions:
        parts.append(
            "DOMAIN DEFINITIONS\n"
            + "\n".join(
                f"- {item}"
                for item in semantic_context.definitions
            )
        )

    if semantic_context.business_rules:
        parts.append(
            "BUSINESS RULES\n"
            + "\n".join(
                f"- {item}"
                for item in semantic_context.business_rules
            )
        )

    if semantic_context.domain_knowledge:
        parts.append(
            "DOMAIN KNOWLEDGE\n"
            + "\n".join(
                f"- {item}"
                for item in semantic_context.domain_knowledge
            )
        )

    if semantic_catalog.semantic_constraints:
        parts.append(
            "SEMANTIC CONSTRAINTS\n"
            + "\n".join(
                f"- {item}"
                for item in semantic_catalog.semantic_constraints
            )
        )

    parts.append("AVAILABLE TABLES")

    for table in semantic_catalog.tables:

        lines = [
            f"Table: {table.name}",
            f"Description: {table.description}",
            "Columns:",
        ]

        for column in table.columns:
            lines.append(
                f"- {column.name}: {column.description or ''}"
            )

        parts.append("\n".join(lines))

    #return my_prompt 
    return "\n\n".join(parts)


def run_results_interpreter(
    query: str,
    llm,
    data_component: ResultsInterpreterData,
    tools: ResultsInterpreterTools,
    background: str | None = None,
):

    prompt = build_prompt(data_component)

    agent_tools = tools.get_agent_tools()

    agent = create_agent(
        model=llm,
        system_prompt=prompt,
        tools=agent_tools,
    )

    user_message = query

    if background:
        user_message = (
            f"BACKGROUND INFORMATION:\n"
            f"{background}\n\n"
            f"USER QUESTION:\n"
            f"{query}"
        )

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_message,
                }
            ]
        }
    )

    return response




In [21]:
data_component = ResultsInterpreterData()
data_component.set_data(raw_data, metadata)

tools = ResultsInterpreterTools()
tools.set_data_component(data_component)

from get_llm_model import azure_llm_if as get_llm 

llm = get_llm()

response = run_results_interpreter(
    query="Which injector has the highest utility? how did you get that answer?",
    llm=llm,
    data_component=data_component,
    tools=tools,
)


m = response['messages'][-1].content
from IPython.display import display, Markdown
display(Markdown(m))


zero temp, seed 42, top_p = 1


The injector with the highest utility is **I1**, with a utility value of **0.780**. 

### How I got this answer:
1. I retrieved the injector utility data, which includes the sum of GAIN across all producers supported by each injector.
2. I identified the injector with the highest utility value from the results.

Key metrics for I1:
- Utility: 0.780
- Number of supported producers: 2
- Maximum GAIN to a single producer: 0.40 (connected to producer P2).

In [50]:
m = response['messages'][-1].content

from IPython.display import display, Markdown

display(Markdown(m))

The injector with the highest utility is **I1**, with a utility value of **0.780**. 

### How I got this answer:
1. I retrieved the injector utility data, which includes the sum of GAIN across all producers supported by each injector.
2. I identified the injector with the highest utility value from the results.

Key metrics for I1:
- Utility: 0.780
- Number of supported producers: 2
- Maximum GAIN to a single producer: 0.40 (connected to producer P2).

In [ ]:
data_component

## Executve summary question

In [52]:
semantic_context

SemanticContext(definitions=['A producer is supported when one or more injectors have non-negligible GAIN to that producer.', 'An injector supports producers when it has non-negligible GAIN to one or more producers.', 'Injector utility is related to the sum of GAIN across the producers supported by that injector.'], business_rules=['GAIN < 0.05 is negligible.', 'A high-utility injector supports one or more producers and has total GAIN across producers approaching 1 in a balanced model.', 'Model quality should be considered when assessing confidence in producer-level interpretations.'], domain_knowledge=['A producer may be supported by injection while only a small fraction of its total liquid production is due to injection.', 'Total producer liquid production may contain contributions from injection, depletion, and pressure changes.', 'total_allocation measures the fraction of producer liquid production attributable to all modeled injectors combined.'])

In [22]:
llm = get_llm()

question= """"Produce a summary report of simulation results. Focus on identifying the best and worse injectors in terms of support, 
the most supported producers and quality metrics for simulation. Is there any evidence of channeling?

Is there any producer with high water cut that is strongly supported?
Is there any producer with low water cut that is supported but the suporting injectors have low injection rates?  
"""

response = run_results_interpreter(
    query= question,
    llm=llm,
    data_component=data_component,
    tools=tools,
    )

m = response['messages'][-1].content

print(100*'=')
print("Question:", question)
display(Markdown(m))
print(100*'=')
print()



zero temp, seed 42, top_p = 1
Question: "Produce a summary report of simulation results. Focus on identifying the best and worse injectors in terms of support, 
the most supported producers and quality metrics for simulation. Is there any evidence of channeling?

Is there any producer with high water cut that is strongly supported?
Is there any producer with low water cut that is supported but the suporting injectors have low injection rates?  



### Summary Report of Simulation Results

#### Best and Worst Injectors in Terms of Support
- **Best Injector**: **I1**
  - **Utility**: 0.780
  - **Number of Supported Producers**: 2
  - **Strongest Connected Producer**: P2 (GAIN = 0.40)
- **Worst Injector**: **I3**
  - **Utility**: 0.151
  - **Number of Supported Producers**: 1
  - **Strongest Connected Producer**: P2 (GAIN = 0.15)

#### Most Supported Producers
- **P1**
  - **Number of Supporting Injectors**: 2 (I1 and I2)
  - **Total Allocation**: 0.55
  - **Current Liquid Production**: 1200.0 units
  - **Water Cut**: 72% (High)
- **P2**
  - **Number of Supporting Injectors**: 2 (I1 and I3)
  - **Total Allocation**: 0.30
  - **Current Liquid Production**: 950.0 units
  - **Water Cut**: 35% (Moderate)

#### Simulation Quality Metrics
- All producers have **good** quality scores:
  - **P4**: Highest quality score (0.9179)
  - **P3**: Lowest quality score (0.8339)

#### Evidence of Channeling
- **P3** has a very high water cut (88%) and is strongly supported by **I1** (PALLOCATION = 0.70). This suggests potential channeling, as a significant fraction of liquid production is water.

---

### Specific Questions

#### 1. Is there any producer with high water cut that is strongly supported?
- **P1**:
  - **Water Cut**: 72% (High)
  - **Strong Support**: Supported by **I1** (PALLOCATION = 0.508) and **I2** (PALLOCATION = 0.432).

#### 2. Is there any producer with low water cut that is supported but the supporting injectors have low injection rates?
- **P4**:
  - **Water Cut**: 20% (Low)
  - **Support**: No supporting injectors (number of supporting injectors = 0). This producer is primarily driven by depletion and pressure changes.

Let me know if you need further analysis or details!

## Scenario-like question
This must be diverted to the scenario analyst. This is just a test 

In [46]:
questions = [
        """
    If I were to shut one injector, which one would be the one that would affect the less possible the oil production? 

    In your answer, briefly explain:
    - which data/tool you used,
    - which metric determines injector utility,
    - the main values that support your conclusion.
    """,

    """
    Which injector has the highest utility?

    In your answer, briefly explain:
    - which data/tool you used,
    - which metric determines injector utility,
    - the main values that support your conclusion.
    """,

    """
    Which injectors meaningfully support producer P1?

    In your answer, briefly explain:
    - which data/tool you used,
    - how you determined whether a connection is meaningful,
    - the GAIN and PALLOCATION values relevant to the conclusion.
    """,

    """
    How dependent is producer P2 on water injection compared with depletion
    and pressure effects?

    In your answer, briefly explain:
    - which data/tool you used,
    - the production contributions you compared,
    - how total allocation relates to your conclusion.
    """,

    """
    Summarize the main CRM results, focusing on injector utility,
    producer support, production-source contributions, and model quality.

    In your answer, briefly explain:
    - which tools/tables you used,
    - the key metrics considered,
    - any limitations or low-confidence conclusions caused by model quality.
    """
]

In [47]:
for question in questions[0:1]:

    response = run_results_interpreter(
    query= question,
    llm=llm,
    data_component=data_component,
    tools=tools,
    )

    m = response['messages'][-1].content

    print(100*'=')
    print("Question:", question)
    display(Markdown(m))
    print(100*'=')
    print()
    
    
    

Question: 
    If I were to shut one injector, which one would be the one that would affect the less possible the oil production? 

    In your answer, briefly explain:
    - which data/tool you used,
    - which metric determines injector utility,
    - the main values that support your conclusion.
    


### Analysis and Conclusion

#### Data and Tools Used:
1. **Injector Utility Summary**:
   - Provides the utility of each injector based on the sum of GAIN across producers.
   - Key metrics: UTILITY, NUMBER_SUPPORTED_PRODUCERS, and MAX_GAIN.
2. **Producer Model Table**:
   - Provides current oil production and PALLOCATION values to estimate the oil production impact of each injector.

#### Metrics:
- **Injector Utility**: Lower utility indicates less overall support to producers.
- **PALLOCATION and Oil Fraction**: Used to estimate the oil production impact of shutting down an injector.

#### Key Values:
1. **Injector Utility Summary**:
   - **I1**: UTILITY = 0.780, supports 2 producers, MAX_GAIN = 0.40 (strongest connection to P2).
   - **I2**: UTILITY = 0.300, supports 1 producer, MAX_GAIN = 0.30 (strongest connection to P1).
   - **I3**: UTILITY = 0.151, supports 1 producer, MAX_GAIN = 0.15 (strongest connection to P2).

2. **Producer Model Table**:
   - **P1**: Current oil production = \(1200 \times 0.28 = 336\) barrels/day.
   - **P2**: Current oil production = \(950 \times 0.65 = 617.5\) barrels/day.
   - **P3**: Current oil production = \(700 \times 0.12 = 84\) barrels/day.
   - **P4**: Current oil production = \(1500 \times 0.80 = 1200\) barrels/day.

#### Conclusion:
- **I3** has the lowest utility (0.151) and supports only one producer (P2) with a MAX_GAIN of 0.15.
- Shutting down **I3** would have the least impact on oil production because:
  - Its contribution to P2's liquid production is minimal.
  - P2's oil fraction (0.65) is high, but the low GAIN and utility of I3 suggest limited influence.

Thus, **I3** is the injector that would affect oil production the least if shut down.

# More manual and hard-coded prompt 
...with more guidance


In [ ]:
results_analyst_prompt = """

"""


Lets load some dummy data to work with. Here we load three tables: injectors, producers, locations

In [3]:
def get_config():
    return None 

# mock of DATAIKU setup  
class DataDrivenStorage:
        
    def __init__( self, config_vars ):
        pass 

    def get_project_dataset(self, project_name=None, filters=None):
        #path =  "../datasets/Demo1/"
        path =  Path("../../datasets/IX5I_4P/") 

        inj, prod, locs = self.fetch_data(path) 
        return inj, prod, locs

    def fetch_data(self,path:Path):
        inj  = pd.read_csv(path / "injectors.csv")
        pinj = pd.read_csv(path / "producers.csv")
        locs = pd.read_csv(path / "locations.csv")
        inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
        inj['DAY']   = inj['DATE'].dt.day
        inj['MONTH'] = inj['DATE'].dt.month
        inj['YEAR']  = inj['DATE'].dt.year
        pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
        pinj['DAY']   = pinj['DATE'].dt.day
        pinj['MONTH'] = pinj['DATE'].dt.month
        pinj['YEAR']  = pinj['DATE'].dt.year


        return inj, pinj, locs


In [4]:
# mock of fetching CRM input data 
inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
display( inj.head(2) )
display( prod.head(2))
display( locs.head(2))

# these tables are used by the visualization system (input data)

,DATE,NAME,WATER_INJECTION_VOLUME,SECTOR,ZONE,SUBZONE,WELL_TYPE,DAY,MONTH,YEAR
0,2015-12-02,I1,0.00,1,WARA,WARA1,Injector,2,12,2015
1,2016-01-02,I1,293.71,1,WARA,WARA1,Injector,2,1,2016


,DATE,NAME,LIQUID_VOLUME,SECTOR,ZONE,SUBZONE,WELL_TYPE,GAS_VOLUME,WATER_VOLUME,OIL_VOLUME,PRESSURE,DAY,MONTH,YEAR
0,2015-12-02,P1,0.000000,1,WARA,WARA1,Producer,0.000000,0.000000,0.000000,0,2,12,2015
1,2016-01-02,P1,910.803528,1,WARA,WARA1,Producer,240.452133,0.000001,910.803528,250,2,1,2016


,NAME,X,Y,SECTOR,ZONE,SUBZONE,WELL_TYPE
0,P1,1240,2040,1,WARA,WARA1,Producer
1,P2,440,1240,1,WARA,WARA1,Producer


In [5]:
# lets create smart data for those tables.
# For a fully operational SmartData object we need 
# 1. the data 
# 2. the semantic models 

# but we can initialize it from the semantic models and pass data later when we have it,
# we can initialize the object with both at the same time.
# every time data is "set" all previous tables are deleted.
# the semantic model doesnt change automatically. if needed, use the provided method. 



#This is a dictionary of table-name: semantic info
from agentic_system.common.known_tables_models import inj_prod_locs_semantic_catalog 
#inj_prod_locs_semantic_catalog
semantic_catalog = SemanticCatalog.model_validate(inj_prod_locs_semantic_catalog)
#type(semantic_catalog)

known_table_models = { t.name: t for t in semantic_catalog.tables } 
df_dict = {'injectors': inj, 'producers': prod , 'locations': locs }
#models = [ TableCard(x) for x in inj_prod_locs_semantic_catalog]

#one option 
smart_data = SmartData()
smart_data.init_from_semantic_models( semantic_catalog.tables )
smart_data.set_data( df_dict )



In [6]:
[x for x in dir(smart_data) if not x.startswith('_') ] 

['catalog_snapshot',
 'clear',
 'clear_derived',
 'conn',
 'execute_sql',
 'get_df',
 'get_single_table_brief_description',
 'get_table_as_df',
 'get_table_names',
 'get_tables_brief_description',
 'get_tables_creation_datetime',
 'init_from_data_and_models',
 'init_from_semantic_models',
 'initialize_from_named_dataframes',
 'register_derived_table',
 'restart_connection',
 'sanitize_df',
 'set_data']

In [7]:
print(smart_data.get_table_names())
print(smart_data.get_tables_brief_description())

smart_data.get_table_as_df('producers')
smart_data.get_single_table_brief_description('producers')

model = smart_data.catalog_snapshot('producers') #(), (['producers','locations'])

model.model_dump() 

['injectors', 'producers', 'locations']
{'injectors': 'Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', 'producers': 'Production time series. Each row contains a dated observation of produced volumes (oil,gas,water) and ratios (opional) for a given producer name, sector and subzone', 'locations': 'Data of well name, sector, well type and coordinates of the named well in each subzone.'}


{'base_tables': [{'name': 'producers',
   'description': 'Production time series. Each row contains a dated observation of produced volumes (oil,gas,water) and ratios (opional) for a given producer name, sector and subzone',
   'kind': 'base',
   'creation_date': '2026-08-26 13:54:56.019571',
   'row_count': 392,
   'columns': [{'name': 'DATE',
     'data_type': 'timestamp',
     'description': 'Production date.',
     'derived_column': False},
    {'name': 'NAME',
     'data_type': 'string',
     'description': 'Producer well identifier.',
     'derived_column': False},
    {'name': 'LIQUID_VOLUME',
     'data_type': 'float',
     'description': 'Total produced liquid.',
     'derived_column': False},
    {'name': 'WATER_VOLUME',
     'data_type': 'float',
     'description': 'Produced water.',
     'derived_column': False},
    {'name': 'GAS_VOLUME',
     'data_type': 'float',
     'description': 'Produced gas.',
     'derived_column': False},
    {'name': 'OIL_VOLUME',
     'data_ty

The key is that smart data can execute sql. 
We can pass the sql directly or get an agent to generate it.
If we use an agent, then we can pass to it the smart tools linked to smart data 
and it will be aware of the tables semantic models. 



In [8]:
result = smart_data.execute_sql(
    """
        SELECT
            NAME,
            SUM(WATER_INJECTION_VOLUME) AS total_injection
        FROM injectors
        GROUP BY NAME
        ORDER BY total_injection DESC
        LIMIT 3
    """)

print(result)


  NAME  total_injection
0   I1       157996.408
1   I2       125508.395
2   I5        96638.092


#### SmartTools

In [9]:
smart_tools = SmartDataTools( smart_data ) 

#these are tools for an agent:
tools = smart_tools.get_agent_tools() 

tools 

[StructuredTool(name='catalog_snapshot', description='Return an LLM-friendly textual snapshot of the data catalog.\n\nThis method is intended to ground agents with the available table\nschemas, descriptions, columns, and relevant metadata before they plan\nor execute data tasks.\n\nParameters\n----------\ninput_tables : None | str | Iterable[str], optional\n    Tables to include in the snapshot.\n\n    - None:\n        Include all tables in the catalog.\n    - str:\n        Include only the table with this name.\n    - Iterable[str]:\n        Include only the listed table names.\n\nReturns\n-------\nstr\n    A structured, readable catalog description suitable for use in\n    planner prompts, executor prompts, and schema-grounded reasoning.', args_schema=<class 'langchain_core.utils.pydantic.catalog_snapshot'>, func=<bound method SmartDataTools.catalog_snapshot of <agentic_system.visualization.smart_data_tools.SmartDataTools object at 0x0000015F06809150>>),
 StructuredTool(name='get_sin

#### An agent to query these tables via natural language 

 - Embelishments 

Before moving to an agent, we need a couple of embelishments.
These are to secure that the SQL code is compliant with the sql-engine running behind the scenes. Also to secure that it is syntactically correct. These embelishments are:

- Constrains
- SQL flavour specific. Here we will use duckdb flavour.

These will enter in the agent prompt later as:

- {idiom}
- {idiom_examples}
- {constraints}



In [10]:
from agentic_system.common.known_idioms import idioms as all_idioms

idiom = 'duckdb'
idiom_rules = all_idioms[ idiom ]

constraints = "".join([f"- {i}\n" for i in semantic_catalog.semantic_constraints])
idiom_context = "".join([f"- {i}: {v}\n" for i,v in idiom_rules.items()])



In [11]:
anayst_prompt_template1d = """
You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database


===============================================================================
Workflow:
===============================================================================
You must:
1. Always call catalog_snapshot as the first step of the workflow.
After catalog_snapshot, check whether all requested concepts map clearly to catalog tables, columns, or known metrics.
If not, ask clarification and do not call sql_* tools.

2. Analyze the question and the information in the catalog and produce a concise PLAN
The PLAN must be concise and must include:
- required source tables
- whether existing derived tables can be reused
- target table names to materialize
- high-level transformation logic, without SQL


3. You MUST ALWAYS record the PLAN in plain text. Only after the PLAN message is sent may you call sql_* tools.
4. Use sql_materialize to create intermediate tables.
5. When multiple output tables are to be produced, proceed sequentially one at a time  
6. Your job finishes once all the target tables are confirmed present (either via initial audit or your materializations).  

===============================================================================
Important:
===============================================================================
- The name of generated tables and columns should reflect the table contents  

- Use lowercase snake_case for table names and column names 
    Example 1: yearly_aggregated_oil_producer_per_subzone
    Example 2: gas_oil_water_cummulated_volumes 

- Be explicit in the detailed description of tables produced 

- Sequential Execution:  If you need to materialize multiple tables, do so one by one, verifying the metadata for each.

===============================================================================
Output
===============================================================================
In each turn you will provide as result one or more tables 

Do not proceed if the question cannot be answered with the available data. 
Instead, ask for clarification 

===============================================================================
SQL generation rules 
===============================================================================
- ALWAYS use **{idiom}** compliant SQL syntax when generating queries.
{idiom_examples}

===============================================================================
Domain constraints
===============================================================================
{constraints}

===============================================================================
Chart-ready output rules
===============================================================================
CHART-READY OUTPUT RULES

For chart/plot/graph requests:

- "plot A by B"
  => return one row per B

- "plot A by B,C"
  => return one row per (B,C)

- "plot A by B,C,D"
  => return one row per (B,C,D)

Rules:
- Preserve all grouping columns.
- Aggregate A at the requested grouping level.
- Use sum by default for additive quantities unless another aggregation is requested.
- If multiple grouping columns together naturally define the chart axis, also create a readable display label column.
- Do not return raw detail rows for grouped chart requests.

Do not return raw detail rows when the user asks for aggregated chart-ready output.

===============================================================================
MANDATORY REUSE RULE:
===============================================================================

After calling catalog_snapshot:

1. If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.

2. You MUST NOT recompute intermediate tables if an equivalent derived table already exists.

3. Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.


Important:
- If a query depends on a table, ensure it has been materialized first.
- Always ask for clarification if the question is ambiguous or cannot be answered with the available data.



"""


prompt = anayst_prompt_template1d.format(idiom = idiom, 
                                    idiom_examples = idiom_context, 
                                    constraints = constraints)


Finally the agent 

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage
from typing import Any, Dict, Generic, List, Iterable, Literal, TypeVar, Union, Optional,TypedDict
from typing_extensions import Self
from agentic_system.common.get_llm import azure_llm_if
from pathlib import Path
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent


In [13]:

from agentic_system.common.structured_responses import AgentTableResponse

llm = get_llm() 
query = "which are the two injectors with the highest cummulated water injection "

agent = create_agent(
            model=llm,
            system_prompt = prompt,
            tools=tools,
            #response_format=ToolStrategy(AgentTableResponse),
        )

response = agent.invoke({
            "messages": [
                {"role": "user", "content": query}
            ]
        })

raw_result = response.get("structured_response")


zero temp, seed 42, top_p = 1


In [14]:
response['messages'][-1]

AIMessage(content='The two injectors with the highest cumulative water injection volumes are:\n\n1. Injector `I1` with a total water injection volume of **157,996.41**.\n2. Injector `I2` with a total water injection volume of **125,508.40**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 2965, 'total_tokens': 3024, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 2688}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 43, 'engine_ttlt_ms': 363, 'pre_inference_ms': 624, 'service_tbt_ms': 6, 'service_ttft_ms': 748, 'service_ttlt_ms': 1064, 'total_duration_ms': 452, 'user_visible_ttft_ms': 124}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_85fa909971', 'id': 'chatcmpl-EH7RwRBJTvw8jzIilkkQUpRhKi8vq', 'service_tier': 

In [15]:
raw_result

In [16]:
#print( response )
#table_name = "top_two_injectors_by_water_injection"
table_name = raw_result.tables[0].table_name 


smart_data.get_table_as_df( table_name )

AttributeError: 'NoneType' object has no attribute 'tables'

# test to load catalogs and context straight from json files 
 

In [1]:
import sys 
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from pathlib import Path 

BASE = Path(r"C:\Work\2026\KOC_phase2\AgenticWaterfloodInsights3\agentic_system\results_interpreter")

semantic_catalog_file = BASE /  "tables_semantic_catalog.json"
semantic_context_file = BASE /  "domain_context.json"
from agentic_system.common.semantic_models import SemanticCatalog,SemanticContext,catalog_from_file,context_from_file

tables_semantic_catalog = catalog_from_file(semantic_catalog_file)
semantic_context = context_from_file(semantic_context_file)
semantic_context.model_dump()


{'definitions': ['A producer is supported when one or more injectors have non-negligible GAIN to that producer.',
  'An injector supports producers when it has non-negligible GAIN to one or more producers.',
  'Injector utility is related to the sum of GAIN across the producers supported by that injector.'],
 'business_rules': ['GAIN < 0.05 is negligible.',
  'A high-utility injector supports one or more producers and has total GAIN across producers approaching 1 in a balanced model.',
  'Model quality must be considered when assessing confidence in producer-level interpretations.'],
 'domain_knowledge': ['A producer may be supported by injection while only a small fraction of its total liquid production is due to injection.',
  'Total producer liquid production may contain contributions from injection, primary depletion, and pressure changes.',
  'total_allocation measures the fraction of producer liquid production attributable to all modeled injectors combined.']}

# Script-based run 

In [2]:
import sys 
sys.path.append("./")
sys.path.append("../")
sys.path.append("../../")


import io
import pandas as pd
from langchain.agents import create_agent
from get_llm_model import azure_llm_if as get_llm 
from pathlib import Path 

from agentic_system.results_interpreter.agent import ResultsInterpreterComponent, ResultsInterpreterConfig, ResultsInterpreterTools, ResultsInterpreterData
from agentic_system.common.semantic_models import SemanticCatalog,SemanticContext,catalog_from_file,context_from_file


imported


## Import data 

In [3]:


c = """
INJECTOR	PRODUCER PALLOCATION	GAIN	GAIN_CLASS  
0	I1	P1	0.508456	0.38   meaningful
1	I2	P1	0.432001	0.30   meaningful
2	I3	P1	0.007279	0.001  negligible
3	I1	P2	0.463442	0.4    meaningful
4	I3	P2	0.226982	0.15   meaningful
"""
q="""	PRODUCER	CORRELATION	VARIANCE_RATIO	QUALITY_SCORE	QUALITY_CLASS
0	P1	0.802377	0.929430	0.864767	good
1	P2	0.929300	1.324738	0.886701	good
2	P3	0.754035	1.020565	0.833949	good
3	P4	0.925126	1.205418	0.917859	good
"""
p="""
  PRODUCER  current_produced_water_fraction  current_produced_oil_fraction  current_volume_liquid_produced  current_liquid_production_due_to_depletion  current_liquid_production_due_to_injection  current_liquid_production_due_to_pressure  pressure_coefficient  total_allocation  number_supporting_injectors
0   P1                             0.72                           0.28                          1200.0                               420.0                              660.0                              120.0                  0.18              0.55                           2
1   P2                             0.35                           0.65                           950.0                               570.0                              285.0                               95.0                  0.10              0.30                           2
2   P3                             0.88                           0.12                           700.0                               140.0                              490.0                               70.0                  0.22              0.70                           1
3   P4                             0.20                           0.80                          1500.0                              1050.0                              300.0                              150.0                  0.08              0.20                           0
"""


# Read the string into a DataFrame
# 'sep=r"\s+"' handles any whitespace (tabs or multiple spaces)
# 'index_col=0' uses the first column (0, 1, 2...) as the row index
connectivity_table  = pd.read_csv(io.StringIO(c), sep=r"\s+", index_col=0)
simulation_quality_table = pd.read_csv(io.StringIO(q), sep=r"\s+", index_col=0)
producer_model_table = pd.read_csv(io.StringIO(p), sep=r"\s+", index_col=0)


print(connectivity_table)
print(simulation_quality_table)
print(producer_model_table.T)


BASE = Path(r"C:\Work\2026\KOC_phase2\AgenticWaterfloodInsights3\agentic_system\results_interpreter")

semantic_catalog_file = BASE /  "tables_semantic_catalog.json"
semantic_context_file = BASE /  "domain_context.json"

tables_semantic_catalog = catalog_from_file(semantic_catalog_file)
semantic_context = context_from_file(semantic_context_file)

metadata = {
    "semantic_catalog": tables_semantic_catalog,
    "semantic_context": semantic_context,
}


raw_data = {
    "connectivity_table": connectivity_table,
    "simulation_quality_table": simulation_quality_table,
    "producer_model_table": producer_model_table,
}





  INJECTOR PRODUCER  PALLOCATION   GAIN  GAIN_CLASS
0       I1       P1     0.508456  0.380  meaningful
1       I2       P1     0.432001  0.300  meaningful
2       I3       P1     0.007279  0.001  negligible
3       I1       P2     0.463442  0.400  meaningful
4       I3       P2     0.226982  0.150  meaningful
  PRODUCER  CORRELATION  VARIANCE_RATIO  QUALITY_SCORE QUALITY_CLASS
0       P1     0.802377        0.929430       0.864767          good
1       P2     0.929300        1.324738       0.886701          good
2       P3     0.754035        1.020565       0.833949          good
3       P4     0.925126        1.205418       0.917859          good
                                                 0      1      2       3
PRODUCER                                        P1     P2     P3      P4
current_produced_water_fraction               0.72   0.35   0.88     0.2
current_produced_oil_fraction                 0.28   0.65   0.12     0.8
current_volume_liquid_produced              1200.0 

In [4]:

def build_prompt(data_component: ResultsInterpreterData) -> str:

    metadata = data_component.metadata

    semantic_catalog: SemanticCatalog = metadata["semantic_catalog"]
    semantic_context: SemanticContext = metadata["semantic_context"]

    parts = [
        """
You are a Results Interpreter specialized in CRM simulation results.

Use the available tools to answer questions about:
- injector utility
- producer support
- injector-producer connectivity
- producer production contributions
- simulation quality

Do not assume that one tool is sufficient.

When the user asks about the effect of injectors on producer oil production,
combine injector-producer connectivity information with producer-level current
production information.

Use PALLOCATION to estimate the fraction of producer liquid attributable to a
specific injector.

For questions about oil impact, combine PALLOCATION with the producer's current
liquid production and current oil fraction.

Use injector utility based on GAIN only when the question is specifically about
injector support/utility, not oil production impact.


Use the supplied semantic definitions as authoritative.
Do not invent alternative meanings for domain-specific metrics.
""".strip()
    ]

    if semantic_context.definitions:
        parts.append(
            "DOMAIN DEFINITIONS\n"
            + "\n".join(
                f"- {item}"
                for item in semantic_context.definitions
            )
        )

    if semantic_context.business_rules:
        parts.append(
            "BUSINESS RULES\n"
            + "\n".join(
                f"- {item}"
                for item in semantic_context.business_rules
            )
        )

    if semantic_context.domain_knowledge:
        parts.append(
            "DOMAIN KNOWLEDGE\n"
            + "\n".join(
                f"- {item}"
                for item in semantic_context.domain_knowledge
            )
        )

    if semantic_catalog.semantic_constraints:
        parts.append(
            "SEMANTIC CONSTRAINTS\n"
            + "\n".join(
                f"- {item}"
                for item in semantic_catalog.semantic_constraints
            )
        )

    parts.append("AVAILABLE TABLES")

    for table in semantic_catalog.tables:

        lines = [
            f"Table: {table.name}",
            f"Description: {table.description}",
            "Columns:",
        ]

        for column in table.columns:
            lines.append(
                f"- {column.name}: {column.description or ''}"
            )

        parts.append("\n".join(lines))

    #return my_prompt 
    return "\n\n".join(parts)

def run_results_interpreter(
    query: str,
    llm,
    data_component: ResultsInterpreterData,
    tools: ResultsInterpreterTools,
    background: str | None = None,
):

    prompt = build_prompt(data_component)

    agent_tools = tools.get_agent_tools()

    agent = create_agent(
        model=llm,
        system_prompt=prompt,
        tools=agent_tools,
    )

    user_message = query

    if background:
        user_message = (
            f"BACKGROUND INFORMATION:\n"
            f"{background}\n\n"
            f"USER QUESTION:\n"
            f"{query}"
        )

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_message,
                }
            ]
        }
    )

    return response



In [5]:
data_component = ResultsInterpreterData()
data_component.set_data(raw_data, metadata)

data_component.metadata

{'semantic_catalog': SemanticCatalog(semantic_constraints=['GAIN and PALLOCATION have different physical meanings and must not be used interchangeably.', "GAIN is injector-centric: it measures the fraction of an injector's injected water recovered as liquid production at a producer.", "PALLOCATION is producer-centric: it measures the fraction of a producer's liquid production attributed to a specific injector.", 'GAIN below 0.05 is considered negligible and must not count as meaningful injector support.', 'For a balanced simulation, the sum of GAIN across producers for each injector must be less than or equal to 1.', 'The sum of PALLOCATION across injectors for each producer must be less than or equal to 1.', 'Produced water fraction and produced oil fraction must sum to 1, subject to numerical tolerance.'], tables=[TableCard(name='connectivity_table', description='CRM injector-producer connectivity results. Each row represents one modeled injector-producer pair.', kind='derived', crea

In [6]:


llm = get_llm()
tools = ResultsInterpreterTools( llm = llm )
tools.set_data_component(data_component)


response = run_results_interpreter(
    query="from the connectivity table determine Which injector has the highest utility? how did you get that answer?",
    llm=llm,
    data_component=data_component,
    tools=tools,
)


m = response['messages'][-1].content
from IPython.display import display, Markdown
display(Markdown(m))

zero temp, seed 42, top_p = 1


The injector with the highest utility is **I1**, with a utility value of **0.780**.

### How I determined this:
1. I retrieved the injector utility data, which includes the sum of GAIN across all producers supported by each injector.
2. I identified the injector with the highest utility value from the results.

Key metrics for **I1**:
- Utility: 0.780
- Number of supported producers: 2
- Maximum GAIN to a single producer: 0.40 (connected to producer P2).

In [8]:
response = run_results_interpreter(
    query="produce a report of simulation results",
    llm=llm,
    data_component=data_component,
    tools=tools,
)


m = response['messages'][-1].content
from IPython.display import display, Markdown
display(Markdown(m))

### Simulation Results Report

#### 1. Injector-Producer Connectivity
The following table summarizes the connectivity between injectors and producers, including the fraction of water injected by an injector that is recovered as liquid production at a producer (GAIN) and the fraction of a producer's liquid production attributed to a specific injector (PALLOCATION).

| INJECTOR | PRODUCER | PALLOCATION | GAIN  | GAIN_CLASS   |
|----------|----------|-------------|-------|--------------|
| I1       | P1       | 0.508456    | 0.380 | meaningful   |
| I2       | P1       | 0.432001    | 0.300 | meaningful   |
| I3       | P1       | 0.007279    | 0.001 | negligible   |
| I1       | P2       | 0.463442    | 0.400 | meaningful   |
| I3       | P2       | 0.226982    | 0.150 | meaningful   |

---

#### 2. Producer-Level Current Production and Support
The table below provides details on the current production state of producers, including liquid production rates, contributions from injection, depletion, and pressure, as well as the number of supporting injectors.

| PRODUCER | Water Fraction | Oil Fraction | Liquid Produced (bbl/day) | Injection Support (bbl/day) | Depletion Support (bbl/day) | Pressure Support (bbl/day) | Total Allocation | Supporting Injectors |
|----------|----------------|--------------|---------------------------|-----------------------------|-----------------------------|----------------------------|------------------|-----------------------|
| P1       | 0.72           | 0.28         | 1200.0                   | 660.0                       | 420.0                       | 120.0                      | 0.55             | 2                     |
| P2       | 0.35           | 0.65         | 950.0                    | 285.0                       | 570.0                       | 95.0                       | 0.30             | 2                     |
| P3       | 0.88           | 0.12         | 700.0                    | 490.0                       | 140.0                       | 70.0                       | 0.70             | 1                     |
| P4       | 0.20           | 0.80         | 1500.0                   | 300.0                       | 1050.0                      | 150.0                      | 0.20             | 0                     |

---

#### 3. Simulation Quality
The following table summarizes the quality of the CRM simulation for each producer, including correlation, variance ratio, and a composite quality score.

| PRODUCER | Correlation | Variance Ratio | Quality Score | Quality Class |
|----------|-------------|----------------|---------------|---------------|
| P1       | 0.802377    | 0.929430       | 0.864767      | good          |
| P2       | 0.929300    | 1.324738       | 0.886701      | good          |
| P3       | 0.754035    | 1.020565       | 0.833949      | good          |
| P4       | 0.925126    | 1.205418       | 0.917859      | good          |

---

#### 4. Injector Utility
The table below summarizes the utility of each injector, including the total GAIN across all supported producers, the number of supported producers, and the strongest producer connection.

| INJECTOR | UTILITY | Supported Producers | Max GAIN | Strongest Connected Producer |
|----------|---------|---------------------|----------|------------------------------|
| I1       | 0.780   | 2                   | 0.40     | P2                           |
| I2       | 0.300   | 1                   | 0.30     | P1                           |
| I3       | 0.151   | 1                   | 0.15     | P2                           |

---

This report provides a comprehensive overview of the simulation results, including injector-producer connectivity, producer-level production and support, simulation quality, and injector utility. Let me know if you need further analysis or specific insights!

# Now script run but a complete component 

In [1]:
import sys 
sys.path.append("./")
sys.path.append("../")
sys.path.append("../../")
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
from typing_extensions import Self
from typing import Any, cast
from get_llm_model import azure_llm_if as get_llm 




import io
import pandas as pd
from langchain.agents import create_agent
from get_llm_model import azure_llm_if as get_llm 
from pathlib import Path 

from agentic_system.results_interpreter.agent import ResultsInterpreterComponent, ResultsInterpreterConfig, ResultsInterpreterTools, ResultsInterpreterData
from agentic_system.common.semantic_models import SemanticCatalog,SemanticContext,catalog_from_file,context_from_file

imported


In [2]:
import io
import pandas as pd

c = """
INJECTOR	PRODUCER PALLOCATION	GAIN	GAIN_CLASS  
0	I1	P1	0.508456	0.38   meaningful
1	I2	P1	0.432001	0.30   meaningful
2	I3	P1	0.007279	0.001  negligible
3	I1	P2	0.463442	0.4    meaningful
4	I3	P2	0.226982	0.15   meaningful
"""
q="""	PRODUCER	CORRELATION	VARIANCE_RATIO	QUALITY_SCORE	QUALITY_CLASS
0	P1	0.802377	0.929430	0.864767	good
1	P2	0.929300	1.324738	0.886701	good
2	P3	0.754035	1.020565	0.833949	good
3	P4	0.925126	1.205418	0.917859	good
"""
p = """
  PRODUCER  current_produced_water_fraction  current_produced_oil_fraction  current_volume_liquid_produced  current_liquid_production_due_to_depletion  current_liquid_production_due_to_injection  current_liquid_production_due_to_pressure  pressure_coefficient  total_allocation  number_supporting_injectors  TAU  TAUP  LO
0 P1 0.72 0.28 1200.0 420.0 660.0 120.0 0.18 0.55 2 4.5 120.0 0.35
1 P2 0.35 0.65 950.0 570.0 285.0 95.0 0.10 0.30 2 12.0 240.0 0.60
2 P3 0.88 0.12 700.0 140.0 490.0 70.0 0.22 0.70 1 1.4 450.0 0.20
3 P4 0.20 0.80 1500.0 1050.0 300.0 150.0 0.08 0.20 0 28.0 800.0 0.75
"""

# Read the string into a DataFrame
# 'sep=r"\s+"' handles any whitespace (tabs or multiple spaces)
# 'index_col=0' uses the first column (0, 1, 2...) as the row index
connectivity_table  = pd.read_csv(io.StringIO(c), sep=r"\s+", index_col=0)
simulation_quality_table = pd.read_csv(io.StringIO(q), sep=r"\s+", index_col=0)
producer_model_table = pd.read_csv(io.StringIO(p), sep=r"\s+", index_col=0)


#print(connectivity_table)
#print(simulation_quality_table)
print(producer_model_table.T)


#raw_data = {
#    "connectivity_table": connectivity_table,
#    "simulation_quality_table": simulation_quality_table,
#    "producer_model_table": producer_model_table,
#}

                                                 0      1      2       3
PRODUCER                                        P1     P2     P3      P4
current_produced_water_fraction               0.72   0.35   0.88     0.2
current_produced_oil_fraction                 0.28   0.65   0.12     0.8
current_volume_liquid_produced              1200.0  950.0  700.0  1500.0
current_liquid_production_due_to_depletion   420.0  570.0  140.0  1050.0
current_liquid_production_due_to_injection   660.0  285.0  490.0   300.0
current_liquid_production_due_to_pressure    120.0   95.0   70.0   150.0
pressure_coefficient                          0.18    0.1   0.22    0.08
total_allocation                              0.55    0.3    0.7     0.2
number_supporting_injectors                      2      2      1       0
TAU                                            4.5   12.0    1.4    28.0
TAUP                                         120.0  240.0  450.0   800.0
LO                                            0.35 

In [3]:
from typing import Any
from pydantic import BaseModel
from typing_extensions  import Self
from agentic_system.common.get_llm import azure_llm_if as get_llm  
from agentic_system.results_interpreter.agent import ResultsInterpreterComponent, ResultsInterpreterConfig, ResultsInterpreterTools, ResultsInterpreterData


class xResultsInterpreterComponent( ):

    def __init__(self, 
                 llm, 
                 config:ResultsInterpreterConfig|None = None, 
                 data:ResultsInterpreterData|None = None, 
                 tools:ResultsInterpreterTools|None = None ):

        self.llm = llm 
        self.config = config or ResultsInterpreterConfig()
        self.data_component = data or  ResultsInterpreterData()
        self.domain_tools = tools or ResultsInterpreterTools()
        self.domain_tools.set_data_component( self.data_component )

    def update_metadata( self, metadata:Any|None = None)->Self:
        self.data_component.update_metadata( metadata )
        return self 

    def update_data( self, raw_data:Any|None = None)->Self:
        self.data_component.update_data( raw_data )
        return self  
   
    def set_data(self, raw_data:Any, metadata:Any|None = None)->Self:
        self.data_component.set_data(raw_data, metadata)
        return self
    
    def old_run( self, query:str, background:str|None = None )->Any:
        message =  "[run] I am a hard-coded runner for CRMResultsAnalyst that creates an agent inplace"
        print(message)

        return message 
        #agent_tools = self.domain_tools.get_agent_tools()
        #prompt = "You help with questions related to CRM results"

        #agent = create_agent(model = self.llm,
        #                     system_prompt= prompt,
        #                     tools=agent_tools
        #                     )

      
        #response = agent.invoke({"messages": [{"role": "user", "content": query}]})


        #return response  

        # need to generate a basic plan 

    def run( self, query:str, background:str|None = None )->Any:

        #prompt = build_prompt(data_component)
        #agent_tools = tools.get_agent_tools()
        #agent = create_agent(
        #    model=llm,
        #    system_prompt=prompt,
        #    tools=agent_tools,
        #)

        #user_message = query
        #if background:
        #    user_message = (
        #        f"BACKGROUND INFORMATION:\n"
        #        f"{background}\n\n"
        #        f"USER QUESTION:\n"
        #        f"{query}"
        #    )

        #response = agent.invoke(
        #    {
        #        "messages": [
        #            {
        #                "role": "user",
        #                "content": user_message,
        #            }
        #        ]
        #    }
        #)
        response = "All good, 200 "
        return response





from pathlib import Path 

BASE = Path(r"C:\Work\2026\KOC_phase2\AgenticWaterfloodInsights3\agentic_system\results_interpreter")

semantic_catalog_file = BASE /  "semantic_catalog.json"
semantic_context_file = BASE /  "semantic_context.json"
from agentic_system.common.semantic_models import SemanticCatalog,SemanticContext,catalog_from_file,context_from_file

tables_semantic_catalog = catalog_from_file(semantic_catalog_file)
semantic_context = context_from_file(semantic_context_file)
semantic_context.model_dump()
    

metadata = {
    "semantic_catalog": tables_semantic_catalog,
    "semantic_context": semantic_context,
}


raw_data = {
    "connectivity_table": connectivity_table,
    "simulation_quality_table": simulation_quality_table,
    "producer_model_table": producer_model_table,
}


llm = get_llm()
interpreter =  xResultsInterpreterComponent( llm )
interpreter.update_data( raw_data )
_= interpreter.update_metadata( metadata )



imported
zero temp, seed 42, top_p = 1


# First batch of objectives for the interpreter 

First, define the cognitive tasks the interpreter should be able to perform. This is the initial set:

- answer direct factual questions about the results;
- compare injectors, producers, or connections;
- identify important patterns such as highly connected injectors, poorly supported producers, stranded/underutilized injectors, injection-dominated producers, 
- depletion-dominated producers; 
- interpret CRM parameters such as GAIN, TAU, TAUP, LO, pressure_coefficient;
- qualify conclusions using model quality;
- produce synthesized summaries that prioritize findings rather than restating tables.


Candidate tools:

- Attributable oil produced to injectors 
- Compute injector efficiency
- Sector and field statistics (total water injected, produced, mean injection volume + std, mean production volume + std ... )
- 

tables 
injector_model: name, sector, inj_volume, num_supported producers, gain_sum, efficiency, attributable_oil_produced, attributable_water_produced,   


In [4]:
import sys 
from langchain.agents import create_agent

sys.path.append("./")
sys.path.append("../")
sys.path.append("../../")


from agentic_system.common.semantic_models import SemanticCatalog,SemanticContext,catalog_from_file,context_from_file
from agentic_system.results_interpreter.semantic_models import ResultsInterpreterSemanticContext 
context_file = r"C:\Work\2026\KOC_phase2\AgenticWaterfloodInsights3\agentic_system\results_interpreter\semantic_context.json"
catalog_file = r"C:\Work\2026\KOC_phase2\AgenticWaterfloodInsights3\agentic_system\results_interpreter\semantic_catalog.json"

semantic_context =  context_from_file( context_file, context_type=ResultsInterpreterSemanticContext )
semantic_catalog =  catalog_from_file( catalog_file )

#1500 
RESULTS_INTERPRETER_PROMPT_TEMPLATE = """
You are a Reservoir Engineer specialized in waterflood surveillance,
history matching, and Capacitance Resistance Models (CRM).

Your role is to interpret existing CRM simulation results and answer
questions about injector-producer connectivity, producer support,
production contributions, CRM parameters, and simulation quality.

You must reason from the available simulation results and semantic context.
Do not merely restate table contents when interpretation is possible.

# Scope

You may:
- answer factual questions about CRM simulation results;
- compare injectors, producers, and injector-producer relationships;
- identify important patterns, rankings, extremes, and anomalies;
- interpret CRM parameters and modeled production contributions;
- assess injector utility and producer support;
- assess confidence in conclusions using simulation-quality information;
- produce synthesized summaries of the simulation results.

Scenario evaluation is outside the scope of this component.
Do not predict the effect of shutting wells, changing injection rates,
or applying hypothetical operational interventions.

# Domain Definitions

{definitions}

# Business Rules

{business_rules}

# Domain Knowledge

{domain_knowledge}

# Interpretation Guidelines

{interpretation_guidelines}

# Available Result Tables

{table_catalog}

# Data Usage

Use the table schemas and column descriptions to determine which data is
required for the user's question.

Do not assume that a question must be answered from a single table.
Combine information from multiple tables when the requested interpretation
depends on concepts distributed across them.

Examples of questions mainly answered from connectivity_table:
- "Which injector has the highest utility?"
- "Which injectors meaningfully support producer P1?"

Examples of questions mainly answered from producer_model_table:
- "Which producers are mainly injection-dominated?"
- "Which producers have the fastest injection response?"

Examples of questions mainly answered from simulation_quality_table:
- "Which producers have the poorest simulation quality?"
- "How good is the overall history match across producers?"

Examples requiring multiple tables:
- "Which injector can be attributed the most current oil production?"
  Combine injector-producer PALLOCATION with producer current liquid production
  and current oil fraction.

- "Which strong injector-producer connections are supported by reliable models?"
  Combine connectivity strength with producer simulation quality.

- "Produce a summary of the most important simulation results."
  Synthesize connectivity, producer support and production drivers, CRM behavior,
  and simulation quality rather than summarizing tables independently.

Retrieve only the information necessary for specific questions, but inspect
the relevant result dimensions when producing broad summaries.

# Reasoning Behavior

Before answering, determine what the user is actually asking to evaluate.

Do not assume that terms such as "best", "worst", "important", "strongest",
or "most effective" always refer to the same metric. Infer the intended
criterion from the question and use the appropriate evidence.

Prefer conclusions supported by multiple relevant metrics when possible.

Distinguish clearly between:
1. direct observations from the simulation results;
2. interpretations of modeled behavior;
3. diagnostic hypotheses.

Never present a diagnostic hypothesis as a confirmed physical mechanism.

When a conclusion depends on model quality, use the available simulation-quality
information to determine how strongly that conclusion should be stated.

# Summary Behavior

When the user asks for a broad summary or report, do not simply summarize
each table independently and do not enumerate every row.

Instead, actively identify the most important findings across the simulation.

Consider, when supported by the available data:
- highly utilized or strongly connected injectors;
- low-utilization injectors;
- well-supported producers;
- poorly supported producers;
- important injector-producer relationships;
- injection-dominated producers;
- depletion-dominated producers;
- pressure-dominated producers;
- unusually fast or slow modeled response behavior;
- unusual CRM parameter values;
- potential diagnostic patterns;
- producers with poor simulation quality;
- important conclusions that should be treated with reduced confidence.

Prioritize:
- extremes;
- rankings;
- anomalies;
- combinations of metrics;
- findings supported by several independent indicators;
- findings with reservoir-engineering or operational significance.

Do not enumerate every injector or producer unless the user explicitly
requests an exhaustive listing.

# Quality and Confidence

Simulation quality must be considered when interpreting producer-level results.

If an important conclusion depends on a producer with poor history-match
quality, report that conclusion with reduced confidence.

Strong connectivity or unusual CRM parameters do not automatically imply
a reliable physical interpretation when the corresponding producer model
has poor simulation quality.

# Tool Usage

Use the available tools to retrieve or compute the information needed to
answer the user's question.

For specific questions, retrieve only the information required.

For broad summaries, retrieve enough information to assess the important
patterns across injectors, producers, connectivity, production contributions,
CRM behavior, and simulation quality.

Do not claim that a report, summary, or analysis has been produced unless
the actual findings are included in the response.

# Response Style

Answer as a reservoir engineer interpreting CRM simulation results.

Be concise but analytical.

Lead with the main conclusion when one is clear, followed by supporting
evidence and relevant caveats.

Use actual values when they materially support the conclusion.
"""
 

def build_system_prompt(
    template: str,
    semantic_catalog: SemanticCatalog,
    semantic_context: ResultsInterpreterSemanticContext,
) -> str:

    def format_list(items: list[str]) -> str:
        if not items:
            return "- None"
        return "\n".join(f"- {item}" for item in items)

    def format_table_catalog(catalog: SemanticCatalog) -> str:
        sections = []

        for table in catalog.tables:
            lines = [
                f"## {table.name}",
                table.description,
                "",
                "Columns:",
            ]

            for column in table.columns:
                description = column.description or ""
                lines.append(
                    f"- {column.name} ({column.data_type}): {description}"
                )

            sections.append("\n".join(lines))

        return "\n\n".join(sections)

    return template.format(
        definitions=format_list(
            semantic_context.definitions
        ),
        business_rules=format_list(
            semantic_context.business_rules
        ),
        domain_knowledge=format_list(
            semantic_context.domain_knowledge
        ),
        interpretation_guidelines=format_list(
            semantic_context.interpretation_guidelines
        ),
        table_catalog=format_table_catalog(
            semantic_catalog
        ),
    )



In [5]:

interpreter =  xResultsInterpreterComponent( get_llm() )
interpreter.update_data( raw_data ).update_metadata( metadata )

interpreter.domain_tools.get_agent_tools()

zero temp, seed 42, top_p = 1


[StructuredTool(name='get_connectivity_table', description='Return CRM injector-producer connectivity results.\n\nEach row represents one modeled injector-producer pair.\n\nImportant column semantics:\n- GAIN:\n  Fraction of water injected in an injector that is recovered as\n  liquid production in the producer. GAIN is injector-centric.\n- PALLOCATION:\n  Fraction of the producer\'s total liquid production attributed to\n  injection from that specific injector. PALLOCATION is producer-centric.\n- GAIN_CLASS:\n  Classification of GAIN. Gains below 0.05 are negligible.\n\nParameters\n----------\nproducer_names:\n    Optional list of producer well names to keep.\n\n    If None:\n        Do not filter by producer. Producers of any name may be returned.\n\n    If provided:\n        Return only rows whose PRODUCER value is one of the names\n        in this list.\n\n    Example:\n        producer_names=["P1", "P2"]\n        returns connectivity rows only for producers P1 and P2.\n\ninjector_

In [20]:
pr = build_system_prompt( RESULTS_INTERPRETER_PROMPT_TEMPLATE,semantic_catalog,semantic_context)

prompt = pr
agent = create_agent(
        model=llm,
        system_prompt=prompt,
        tools=interpreter.domain_tools.get_agent_tools(),
    )

query = "summarize results. Focus in potential channeling and stranded injectors and unsupported producers"
query = "Identify unsupported producers, list also the 2 most supported ones. Why?"
query = "Rank producers based on water cut, the smallest the better. Among those, highlight the most supported ones."
query = "are the injectors supporting low utility producers? namely, those producers with high watercut?"

query = """Which injector (one) can I shut while affecting the less possible the total oil produced. State your reasoning, explain your answer, focus on injector efficiency not utility" \
Can you estimate the percent drop in oil production and the percent drop in water injected if I shut in that injector?  
"""


query = "I need to inject less water but want to affect oil production the less possible, what can i do?"

query = """Rank the injectors based on the percentage decrease in total oil production that would result from shutting them down"""

query1 = """Rank the injectors based on their efficiency, from low to high. Produce a table of efficiency and percent of water injected and for each, estimate
the amount of oil production decrease (percent) that would occur if I shut those injectors """


user_message = query1
response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_message,
                }
            ]
        }
    )




In [24]:
!pip install rich 

  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)

   ------------- -------------------------- 1/3 [markdown-it-py]
   ------------- -------------------------- 1/3 [markdown-it-py]
   ------------- -------------------------- 1/3 [markdown-it-py]
   -------------------------- ------------- 2/3 [rich]
   -------------------------- ------------- 2/3 [rich]
   -------------------------- ------------- 2/3 [rich]
   -------------------------- ------------- 2/3 [rich]
   ---------------------------------------- 3/3 [rich]




[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from IPython.display import display, Markdown

# Get the content from your LLM
raw_content = response['messages'][-1].content

# Replace the block LaTeX notation so Jupyter understands it
fixed_content = raw_content.replace(r'\[', '$$').replace(r'\]', '$$')

# Render it beautifully
display(Markdown(fixed_content))


### Calculations

#### Step 1: Injector Efficiency
Using the formula:
$$
\text{Efficiency}_i = \sum_p (\text{GAIN}_{ip} \times \text{Producer Utility}_p)
$$
- **I1**:
  - \( \text{GAIN}_{I1,P1} = 0.380 \), \( \text{Producer Utility}_{P1} = 0.28 \)
  - \( \text{GAIN}_{I1,P2} = 0.400 \), \( \text{Producer Utility}_{P2} = 0.65 \)
  - Efficiency = \( (0.380 \times 0.28) + (0.400 \times 0.65) = 0.1064 + 0.26 = 0.3664 \)

- **I2**:
  - \( \text{GAIN}_{I2,P1} = 0.300 \), \( \text{Producer Utility}_{P1} = 0.28 \)
  - Efficiency = \( 0.300 \times 0.28 = 0.084 \)

- **I3**:
  - \( \text{GAIN}_{I3,P2} = 0.150 \), \( \text{Producer Utility}_{P2} = 0.65 \)
  - Efficiency = \( 0.150 \times 0.65 = 0.0975 \)

#### Step 2: Percent of Water Injected
The total water injected by each injector is proportional to:
$$
\text{Water Injected}_i = \sum_p (\text{GAIN}_{ip} \times \text{Liquid Production}_p)
$$
- **I1**:
  - \( \text{GAIN}_{I1,P1} = 0.380 \), \( \text{Liquid Production}_{P1} = 1200 \)
  - \( \text{GAIN}_{I1,P2} = 0.400 \), \( \text{Liquid Production}_{P2} = 950 \)
  - Water Injected = \( (0.380 \times 1200) + (0.400 \times 950) = 456 + 380 = 836 \)

- **I2**:
  - \( \text{GAIN}_{I2,P1} = 0.300 \), \( \text{Liquid Production}_{P1} = 1200 \)
  - Water Injected = \( 0.300 \times 1200 = 360 \)

- **I3**:
  - \( \text{GAIN}_{I3,P2} = 0.150 \), \( \text{Liquid Production}_{P2} = 950 \)
  - Water Injected = \( 0.150 \times 950 = 142.5 \)

Total Water Injected = \( 836 + 360 + 142.5 = 1338.5 \)

Percent of Water Injected:
- **I1**: \( \frac{836}{1338.5} \times 100 = 62.45\% \)
- **I2**: \( \frac{360}{1338.5} \times 100 = 26.90\% \)
- **I3**: \( \frac{142.5}{1338.5} \times 100 = 10.65\% \)

#### Step 3: Oil Production Decrease
The oil production decrease is approximated as:
$$
\text{Oil Decrease}_i = \sum_p (\text{GAIN}_{ip} \times \text{Producer Utility}_p \times \text{Liquid Production}_p)
$$
- **I1**:
  - \( (0.380 \times 0.28 \times 1200) + (0.400 \times 0.65 \times 950) = 127.68 + 247 = 374.68 \)
  - Percent Decrease = \( \frac{374.68}{(1200 \times 0.28 + 950 \times 0.65)} \times 100 = 374.68 / 847.5 \times 100 = 44.2\% \)

- **I2**:
  - \( (0.300 \times 0.28 \times 1200) = 100.8 \)
  - Percent Decrease = \( \frac{100.8}{(1200 \times 0.28 + 950 \times 0.65)} \times 100 = 100.8 / 847.5 \times 100 = 11.9\% \)

- **I3**:
  - \( (0.150 \times 0.65 \times 950) = 92.625 \)
  - Percent Decrease = \( \frac{92.625}{(1200 \times 0.28 + 950 \times 0.65)} \times 100 = 92.625 / 847.5 \times 100 = 10.9\% \)

### Ranked Table
| Injector | Efficiency | Percent of Water Injected | Estimated Oil Production Decrease (%) |
|----------|------------|---------------------------|---------------------------------------|
| I3       | 0.0975     | 10.65%                   | 10.9%                                |
| I2       | 0.0840     | 26.90%                   | 11.9%                                |
| I1       | 0.3664     | 62.45%                   | 44.2%                                |

This table ranks the injectors from lowest to highest efficiency and provides the associated metrics.

In [9]:
from IPython.display import display, Markdown

# Print the raw text (optional)
print(response['messages'][-1].content)

# Render it beautifully with formatting (headings, bold, lists, etc.)
display(Markdown(response['messages'][-1].content))



The injector that can be shut while affecting the total oil production the least is **I3**. Here's the reasoning:

### Reasoning:
1. **Injector Efficiency**:
   - Injector efficiency is related to the sum of the producer utility (oil fraction) times the GAIN for all producers connected to the injector.
   - For I3:
     - It has a GAIN of 0.15 to P2, which has a high oil fraction of 0.65.
     - The other connection to P1 has a negligible GAIN of 0.001, contributing almost nothing to efficiency.
   - This makes I3 the least efficient injector compared to I1 and I2, which have stronger connections to producers with higher oil fractions.

2. **Impact on Oil Production**:
   - The oil production attributable to I3 can be estimated as:
     \[
     \text{Attributable Oil} = \sum (\text{GAIN}_{ip} \times \text{Producer Oil Fraction} \times \text{Producer Liquid Rate})
     \]
     - For P2: \( 0.15 \times 0.65 \times 950 = 92.625 \, \text{bbl/day} \)
     - For P1: \( 0.001 \times 0.28 \tim

The injector that can be shut while affecting the total oil production the least is **I3**. Here's the reasoning:

### Reasoning:
1. **Injector Efficiency**:
   - Injector efficiency is related to the sum of the producer utility (oil fraction) times the GAIN for all producers connected to the injector.
   - For I3:
     - It has a GAIN of 0.15 to P2, which has a high oil fraction of 0.65.
     - The other connection to P1 has a negligible GAIN of 0.001, contributing almost nothing to efficiency.
   - This makes I3 the least efficient injector compared to I1 and I2, which have stronger connections to producers with higher oil fractions.

2. **Impact on Oil Production**:
   - The oil production attributable to I3 can be estimated as:
     \[
     \text{Attributable Oil} = \sum (\text{GAIN}_{ip} \times \text{Producer Oil Fraction} \times \text{Producer Liquid Rate})
     \]
     - For P2: \( 0.15 \times 0.65 \times 950 = 92.625 \, \text{bbl/day} \)
     - For P1: \( 0.001 \times 0.28 \times 1200 = 0.336 \, \text{bbl/day} \)
     - Total attributable oil = \( 92.625 + 0.336 = 92.961 \, \text{bbl/day} \).

   - The total oil production across all producers is:
     \[
     \text{Total Oil} = \sum (\text{Producer Oil Fraction} \times \text{Producer Liquid Rate})
     \]
     - \( (0.28 \times 1200) + (0.65 \times 950) + (0.12 \times 700) + (0.80 \times 1500) = 336 + 617.5 + 84 + 1200 = 2237.5 \, \text{bbl/day} \).

   - Percent drop in oil production if I3 is shut:
     \[
     \text{Percent Drop} = \frac{\text{Attributable Oil}}{\text{Total Oil}} \times 100 = \frac{92.961}{2237.5} \times 100 \approx 4.15\%.
     \]

3. **Impact on Water Injection**:
   - The total water injected by I3 can be estimated as:
     \[
     \text{Injected Volume} = \sum (\text{GAIN}_{ip} \times \text{Producer Liquid Rate})
     \]
     - For P2: \( 0.15 \times 950 = 142.5 \, \text{bbl/day} \)
     - For P1: \( 0.001 \times 1200 = 1.2 \, \text{bbl/day} \)
     - Total injected volume = \( 142.5 + 1.2 = 143.7 \, \text{bbl/day} \).

   - Total water injected across all injectors is not directly provided, but the percent drop in water injection can be approximated as the fraction of I3's contribution to the total injection.

### Summary:
- **Injector to shut**: I3.
- **Estimated percent drop in oil production**: ~4.15%.
- **Estimated percent drop in water injected**: Proportional to I3's contribution (143.7 bbl/day) relative to the total injection, which needs further data for precise calculation.

# Final agent

Running everything from the library except the data loading and pre-processing

In [ ]:
from dataclasses import dataclass
from typing import Any
from pydantic import BaseModel
from typing_extensions  import Self
from agentic_system.common.get_llm import azure_llm_if as get_llm  
from agentic_system.results_interpreter.agent import ResultsInterpreterComponent, ResultsInterpreterConfig, ResultsInterpreterTools, ResultsInterpreterData # type: ignore
from agentic_system.results_interpreter.prompts import RESULTS_INTERPRETER_PROMPT_TEMPLATE 
 
@dataclass 
class ResultsInterpreterConfig( ):
    promp_template: str | None = RESULTS_INTERPRETER_PROMPT_TEMPLATE 


def get_default_intertpreter( ):
    config = ResultsInterpreterConfig( promp_template=RESULTS_INTERPRETER_PROMPT_TEMPLATE)
    llm = get_llm() 
    
 

class xResultsInterpreterComponent( ):

        

    def __init__(self, 
                 llm, 
                 config:ResultsInterpreterConfig|None = None, 
                 data:ResultsInterpreterData|None = None, 
                 tools:ResultsInterpreterTools|None = None ):

        self.llm = llm 
        self.config = config or ResultsInterpreterConfig()
        self.data_component = data or  ResultsInterpreterData()
        self.domain_tools = tools or ResultsInterpreterTools()
        self.domain_tools.set_data_component( self.data_component )

    def update_metadata( self, metadata:Any|None = None)->Self:
        self.data_component.update_metadata( metadata )
        return self 

    def update_data( self, raw_data:Any|None = None)->Self:
        self.data_component.update_data( raw_data )
        return self  
   
    def set_data(self, raw_data:Any, metadata:Any|None = None)->Self:
        self.data_component.set_data(raw_data, metadata)
        return self

    def _build_system_prompt(self,
        #template: str,
        #semantic_catalog: SemanticCatalog,
        #semantic_context: ResultsInterpreterSemanticContext,
    ) -> str:

        def format_list(items: list[str]) -> str:
            if not items:
                return "- None"
            return "\n".join(f"- {item}" for item in items)

        def format_table_catalog(catalog: SemanticCatalog) -> str:
            sections = []

            for table in catalog.tables:
                lines = [
                    f"## {table.name}",
                    table.description,
                    "",
                    "Columns:",
                ]

                for column in table.columns:
                    description = column.description or ""
                    lines.append(
                        f"- {column.name} ({column.data_type}): {description}"
                    )

                sections.append("\n".join(lines))

            return "\n\n".join(sections)

        template = self.config.promp_template 
        semantic_catalog = self.data_component.metadata['semantic_catalog'] # type: ignore
        semantic_context = self.data_component.metadata['semantic_context'] # type: ignore

        #semantic_catalog = self.data_component.metadata[]

        return template.format( # type: ignore
            definitions=format_list(
                semantic_context.definitions
            ),
            business_rules=format_list(
                semantic_context.business_rules
            ),
            domain_knowledge=format_list(
                semantic_context.domain_knowledge
            ),
            interpretation_guidelines=format_list(
                semantic_context.interpretation_guidelines
            ),
            table_catalog=format_table_catalog(
                semantic_catalog
            ),
        )



    def run( self, query:str, background:str|None = None )->Any:

        #prompt = build_prompt(data_component)
        #agent_tools = tools.get_agent_tools()
        #agent = create_agent(
        #    model=llm,
        #    system_prompt=prompt,
        #    tools=agent_tools,
        #)

        #user_message = query
        #if background:
        #    user_message = (
        #        f"BACKGROUND INFORMATION:\n"
        #        f"{background}\n\n"
        #        f"USER QUESTION:\n"
        #        f"{query}"
        #    )

        #response = agent.invoke(
        #    {
        #        "messages": [
        #            {
        #                "role": "user",
        #                "content": user_message,
        #            }
        #        ]
        #    }
        #)
        response = "All good, 200 "
        return response


def load_interpreter_data_mock():


    from pathlib import Path 

    BASE = Path(r"C:\Work\2026\KOC_phase2\AgenticWaterfloodInsights3\agentic_system\results_interpreter")

    semantic_catalog_file = BASE /  "semantic_catalog.json"
    semantic_context_file = BASE /  "semantic_context.json"
    from agentic_system.common.semantic_models import SemanticCatalog,SemanticContext,catalog_from_file,context_from_file

    tables_semantic_catalog = catalog_from_file(semantic_catalog_file)
    semantic_context = context_from_file(semantic_context_file)
    semantic_context.model_dump()
        

    metadata = {
        "semantic_catalog": tables_semantic_catalog,
        "semantic_context": semantic_context,
    }


    raw_data = {
        "connectivity_table": connectivity_table,
        "simulation_quality_table": simulation_quality_table,
        "producer_model_table": producer_model_table,
    }

    return raw_data, metadata 


llm = get_llm()
raw_data, metadata  = load_interpreter_data_mock()

interpreter =  xResultsInterpreterComponent( llm )
interpreter.update_data( raw_data )
_= interpreter.update_metadata( metadata )



zero temp, seed 42, top_p = 1


In [23]:
metadata.keys()

dict_keys(['semantic_catalog', 'semantic_context'])

In [ ]:
response = run_results_interpreter(
    query="Which injector has the highest utility? how did you get that answer?",
    llm=llm,
    data_component=data_component,
    tools=tools,
)

print(response)

In [38]:
from get_llm_model import azure_llm_if as get_llm 

llm = get_llm()

data_component = ResultsInterpreterData()
data_component.set_data(raw_data, metadata)

tools = ResultsInterpreterTools( llm = llm )
tools.set_data_component(data_component)



response = run_results_interpreter(
    query="Which injector has the highest utility? how did you get that answer?",
    llm=llm,
    data_component=data_component,
    tools=tools,
)

print(response)

zero temp, seed 42, top_p = 1
{'messages': [HumanMessage(content='Which injector has the highest utility? how did you get that answer?', additional_kwargs={}, response_metadata={}, id='ee2951da-f050-45ee-af0f-1d27a7b8821f'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 1133, 'total_tokens': 1147, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1024}, 'latency_checkpoint': {'engine_tbt_ms': 9, 'engine_ttft_ms': 68, 'engine_ttlt_ms': 195, 'pre_inference_ms': 82, 'service_tbt_ms': 9, 'service_ttft_ms': 178, 'service_ttlt_ms': 299, 'user_visible_ttft_ms': 96}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_85fa909971', 'id': 'chatcmpl-EMY8HG1jxV9nXLyIdKirtahikJzJZ', 'service_tier': 'default', 'prompt_filter_

In [34]:
response['messages'][-1].content

"It seems that the injector summary data is not currently available. To determine which injector has the highest utility, I would need access to the GAIN values for each injector across all producers. This information is typically derived from the connectivity table.\n\nIf you can provide the connectivity table or any relevant data, I can calculate the injector utility and identify the injector with the highest utility. Let me know how you'd like to proceed!"

# The DataAnalystComponent 
Basically encloses in one class all the previous functionality.
You need to configure it (semantic, idioms) and call "run" on it. 
It should take care of everything without intervention.


In [17]:

from dataclasses import dataclass

from agentic_system.common.base_domain_tools import BaseDomainTools
from agentic_system.common.structured_responses import TaskResult
from agentic_system.visualization.smart_data import SmartData
from agentic_system.visualization.smart_data_tools import SmartDataTools
from agentic_system.common.structured_responses import *
from agentic_system.visualization.prompts import anayst_prompt_template

@dataclass 
class DataAnalystConfig:
    
    prompt_template :str# = 'anayst_prompt_template' #depends on idioms, constraints, etc.
    prompt: str | None = None     
    use_structured_output : bool = True 

    def build_prompt( self, few_shot_examples = None , background = None ):
        '''
        builds the data_analyst prompt using all the semantics + 
        few_shot examples if any and background info if any
        '''
        pass 


class DataAnalystComponent:
    agent_name = "data_analysis"

    def __init__(
            self,
            llm: Any,
            config: DataAnalystConfig, # | None = None,
            smart_data: SmartData | None = None,
            domain_tools: BaseDomainTools | None = None 
        ):
        self.llm = llm
        self._smart_data = smart_data if smart_data is not None else SmartData()
        self.config = config #if config #is not None else SQLAnalystConfig()

        self.tools_object = SmartDataTools(self._smart_data)
     
        self.domain_tools = domain_tools 
        self.tools = self.tools_object.get_tools()

        if domain_tools:
            self.tools = self.tools + self.domain_tools.get_agent_tools() # type: ignore

    @property
    def prompt(self) -> str:
        return self.config.prompt # type: ignore

    @property
    def smart_data(self) -> SmartData:
        return self._smart_data
    
    def init_semantic_models(self,semantic_catalog, idiom_rules, idiom = 'duckdb'): 
        
        
        # the tables
        #first set the semantic model, when data arrives later we set the new data. No tools or anything will need update
        #this has no data, just semantic models and we dont know the size of the tables 
        self._smart_data.init_from_semantic_models( semantic_catalog.tables )

        # the base prompt (without history, which could be added)
        constraints = "".join([f"- {i}\n" for i in semantic_catalog.semantic_constraints])
        idiom_context = "".join([f"- {i}: {v}\n" for i,v in idiom_rules.items()])
        self.config.prompt = self.config.prompt_template.format(idiom = idiom, 
                                    idiom_examples = idiom_context, 
                                    constraints = constraints)

    def set_data(self,df_dict: Dict[str,pd.DataFrame]):
         self._smart_data.set_data(df_dict) 

    def run(self, query: str, facts_context : str | None  = None) -> TaskResult | str :

        prompt = self.prompt
        if  facts_context:
            prompt = prompt + f"\nCONVERSATION FACTS:\n{facts_context}" 


        response_format = ToolStrategy(AgentTableResponse) if self.config.use_structured_output else None 

        agent = create_agent(
            model=self.llm,
            system_prompt = prompt,
            tools=self.tools,
            response_format=response_format,
        )

        response = agent.invoke({
            "messages": [
                {"role": "user", "content": query}
            ]
        })

        if not response_format:
            #return response 
            return response['messages'][-1].content 


        raw_result = response.get("structured_response")

        if raw_result is None:
            raw_results = []
        elif isinstance(raw_result, list):
            raw_results = raw_result
        else:
            raw_results = [raw_result]

        cheap_parts: list[str] = []
        data_results: list[DataFrameResult] = []

        for raw in raw_results:
            text = getattr(raw, "text", None)
            if text:
                cheap_parts.append(text)

            tables = getattr(raw, "tables", []) or []

            for table in tables:
                table_name = getattr(table, "table_name", None)
                description = getattr(table, "description", None)

                if not table_name:
                    continue

                cheap_parts.append(
                    f"{self.agent_name} agent created and stored the table `{table_name}`: {description or 'No description provided.'}"
                )

                df = self._smart_data.get_table_as_df(table_name)

                data_results.append(
                    DataFrameResult(
                        table_name=table_name,
                        description=description,
                        dataframe=df,
                    )
                )

                if df.shape[0] < 10 and df.shape[1] < 4:
                    cheap_parts.append(
                        f"Table `{table_name}` contents:\n{df.to_string(index=False)}"
                    )

        cheap_output = "\n\n".join(cheap_parts) if cheap_parts else (
            "Data analyst completed, but no textual summary or table metadata was returned."
        )

        return TaskResult(
            agent=self.agent_name,
            instruction=query,
            cheap_output=cheap_output,
            raw_results=raw_results,
            data_results=data_results,
        )


analyst = DataAnalystComponent(llm=llm, 
                               config=DataAnalystConfig(prompt_template=anayst_prompt_template, 
                                                        use_structured_output=True))


analyst.init_semantic_models(semantic_catalog, idiom_rules, idiom = 'duckdb')
analyst.set_data( df_dict )


        

In [18]:
query = "which are the two injectors with the highest cummulated water injection "

response = analyst.run( query )


===== PLAN (TOOL) =====
1. Source table: injectors
2. Aggregate WATER_INJECTION_VOLUME by NAME to calculate the cumulative water injection for each injector.
3. Sort the results in descending order of cumulative water injection.
4. Select the top two injectors with the highest cumulative water injection.
5. Materialize the result as a table named top_two_injectors_by_water_injection.



In [19]:
response

TaskResult(agent='data_analysis', instruction='which are the two injectors with the highest cummulated water injection ', cheap_output='data_analysis agent created and stored the table `top_two_injectors_by_water_injection`: This table contains the top two injectors with the highest cumulative water injection volumes. It includes the injector well identifier (NAME) and the total water injection volume (total_water_injection).\n\nTable `top_two_injectors_by_water_injection` contents:\nNAME  total_water_injection\n  I1             157996.408\n  I2             125508.395', raw_results=[AgentTableResponse(agent='analyst', clarification=None, user_query='which are the two injectors with the highest cummulated water injection', tables=[TableItemAgentResponse(table_name='top_two_injectors_by_water_injection', description='This table contains the top two injectors with the highest cumulative water injection volumes. It includes the injector well identifier (NAME) and the total water injection 

# End 

In [ ]:

def init_visualization_system( llm ):
   

    vis_system = VisualizationAgenticSystem( llm )


    idiom = 'duckdb'
    idiom_rules = all_idiom_rules[idiom]
    semantic_catalog_model = SemanticCatalog.model_validate( semantic_catalog )

    analyst = vis_system.data_analyst_component
    analyst.init_semantic_models( semantic_catalog_model,idiom_rules)
    



    return vis_system


llm = azure_llm_if()
vis_system = init_visualization_system(llm)


# data changes
# this mocks data comming from the UI
# so we just update tge analyst 
inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
vis_system.data_analyst_component.set_data( {'injectors':inj, 
                                             'producers':prod, 
                                             'locations': locs } )



In [ ]:
#from visualization_system.visualization_backend.analyst.analyst_backend import VisualizationAgenticSystem


In [ ]:

# these are just mocks 


from visualization_system.visualization_backend.analyst.analyst_backend import VisualizationAgenticSystem
 
def get_config():
    return None 

class DataDrivenStorage:
        
    def __init__( self, config_vars ):
        pass 

    def get_project_dataset(self, project_name=None, filters=None):
        #path =  "../datasets/Demo1/"
        path =  Path("../datasets/IX5I_4P/") 

        inj, prod, locs = self.fetch_data(path) 
        return inj, prod, locs

    def fetch_data(self,path:Path):
        inj  = pd.read_csv(path / "injectors.csv")
        pinj = pd.read_csv(path / "producers.csv")
        locs = pd.read_csv(path / "locations.csv")
        inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
        inj['DAY']   = inj['DATE'].dt.day
        inj['MONTH'] = inj['DATE'].dt.month
        inj['YEAR']  = inj['DATE'].dt.year
        pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
        pinj['DAY']   = pinj['DATE'].dt.day
        pinj['MONTH'] = pinj['DATE'].dt.month
        pinj['YEAR']  = pinj['DATE'].dt.year


        return inj, pinj, locs

def init_visualization_system( llm ):
   

    vis_system = VisualizationAgenticSystem( llm )


    idiom = 'duckdb'
    idiom_rules = all_idiom_rules[idiom]
    semantic_catalog_model = SemanticCatalog.model_validate( semantic_catalog )

    analyst = vis_system.data_analyst_component
    analyst.init_semantic_models( semantic_catalog_model,idiom_rules)
    



    return vis_system


llm = azure_llm_if()
vis_system = init_visualization_system(llm)


# data changes
# this mocks data comming from the UI
# so we just update tge analyst 
inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
vis_system.data_analyst_component.set_data( {'injectors':inj, 
                                             'producers':prod, 
                                             'locations': locs } )





In [ ]:

query2 = """
List the 5 wells with the highest water cut in current date
"""



#this is what the presenter consumes 
execution_state = vis_system.run( query2 )


In [ ]:

from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
print(ui_items)

for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 




In [ ]:
query2 = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production since year 2015 for all the wells 
"""

planner = vis_system.planner_component
plan = planner.run( query )

In [ ]:
pprint.pprint( plan.model_dump())


In [ ]:
direct_answer = vis_system.direct_answer_component

direct_answer.run(  plan.tasks[0].instruction  ).data_results

In [ ]:

analyst = vis_system.data_analyst_component
task_results = [ analyst.run(task.instruction) for task in plan.tasks[1:] ]



In [ ]:
task_results

In [ ]:
from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = [ presenter.process_single_task_result(result)[0] for result in task_results ] 





In [ ]:
ui_items

In [ ]:


for item in ui_items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 


In [ ]:
query = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production for each year since year 2015 for all the wells 
"""

state = vis_system.run( query )


In [ ]:
presenter = PresenterComponent4( llm )

ui_items = presenter.run(state)

In [ ]:
ui_items.items

In [ ]:
for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 

In [ ]:
state 

In [ ]:
from visualization_system.visualization_backend.analyst.analyst_backend import DataFrameResult, ExecutorState, PresenterConfig, PresenterResponse, SubInstructions, TableResponseProcessor, TaskResult, TextResult, UIItem
  


In [ ]:
from dataclasses import dataclass
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')
import warnings

from typing import Any, Dict, Generic, List, Iterable, Literal, TypeVar, Union, Optional,TypedDict
from typing_extensions import Self
   
from uuid import uuid4
 
from pydantic import BaseModel, Field 
from visualization_system.visualization_backend.get_llm_model import azure_llm_if
from pathlib import Path
from langchain.agents.structured_output import ToolStrategy
from langgraph.graph import StateGraph, END

from langchain.agents import create_agent

from visualization_system.common.base_plan import PlannerConfig
#from visualization_system.visualization_backend.analyst.analyst_system import PlannerConfig, DirectAnswerConfig
from visualization_system.visualization_backend.analyst.analyst_models import TableItemAgentResponse, VisualizationSystemPlanner, VisualizationSystemTask, VisualizationSystemPlan 
from visualization_system.visualization_backend.analyst.smart_data import SmartData
from visualization_system.visualization_backend.analyst.smart_data_tools import SmartDataTools
from visualization_system.visualization_backend.analyst.analyst_system import SQLAnalystConfig

from visualization_system.visualization_backend.analyst.prompts import visualization_planner_prompt3 
from visualization_system.visualization_backend.analyst.prompts import anayst_prompt_template 
from visualization_system.visualization_backend.analyst.prompts import chart_agent_prompt
from visualization_system.visualization_backend.analyst.prompts import small_table_prompt
from visualization_system.visualization_backend.analyst.prompts import split_subinstructions_prompt

 


import pandas as pd, re, json  
from langchain_core.messages import SystemMessage, HumanMessage
from visualization_system.visualization_backend.global_models import UIState



In [ ]:
from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4

CHART_AGENT_PROMPTV4 = """
You are a chart planning agent.

You receive:
- user query
- table summaries
- column names, roles, cardinality, and descriptions

Return a JSON plan with:
- zero or more ordered preprocess operations 
- exactly ONE plot step. 
- ONE plot step (see <plot_tools> below) must be in the plan regardless of whether there are or not preprocess operations

 
PREPROCESSING

The preprocess field is an ordered list of operations applied before plotting.

Supported operations:

1. create_combined_category
   Creates a category column from two existing columns.

Args:
{
"operation": "create_combined_category",
"args": {
"col1": "<column>",
"col2": "<column>",
"new_col": "<new_column>",
"sep": " / "
}
}

2. create_date_bucket
   Creates a date grouping column.

Args:
{
"operation": "create_date_bucket",
"args": {
"date_col": "<date_column>",
"bucket": "D|W|M|Q|Y",
"new_col": "<new_column>"
}
}

3. filter_rows
   Keeps rows matching one or more conditions.

Args:
{
"operation": "filter_rows",
"args": {
"filters": [
{
"column": "<column>",
"operator": "==|!=|>|>=|<|<=|in|not_in",
"value": "<value_or_list>"
}
]
}
}

4. aggregate
   Groups and aggregates the data before plotting.

Args:
{
"operation": "aggregate",
"args": {
"group_by": ["<column>", "..."],
"metrics": {
"<numeric_column>": "sum|mean|median|min|max|count|nunique"
}
}
}

5. sort_rows
   Sorts the rows.

Args:
{
"operation": "sort_rows",
"args": {
"sort_by": "<column_or_list>",
"ascending": true|false
}
}

6. limit_rows
   Keeps only the first N rows.

Args:
{
"operation": "limit_rows",
"args": {
"n": <integer>
}
}

7. select_columns
   Keeps only selected columns.

Args:
{
"operation": "select_columns",
"args": {
"columns": ["<column>", "..."]
}
}

8. select_top_entities
   Selects the top or bottom entities using a metric.

Use keep_all_rows = true when the ranking period is only used to identify entities, but the final chart needs all rows for those entities.

Args:
{
"operation": "select_top_entities",
"args": {
"entity_col": "<entity_column>",
"metric_col": "<numeric_column>",
"n": <integer>,
"aggregate": "sum|mean|median|min|max|count|nunique",
"ascending": true|false,
"filters": [],
"keep_all_rows": true|false
}
}

Rules:

* Use preprocess only when the input table is not already ready for plotting.
* Operations are executed in the listed order.
* Do not invent columns.
* Prefer the smallest number of operations needed.
* Aggregation, filtering, ranking, date bucketing and limiting should be done in preprocess rather than in the plotting tool.



PLOT TOOLS



Args:
{
  "columns": ["<column>", "..."],
  "sort_by": null | "<column>",
  "sort_order": "asc|desc",
  "limit": null | <integer>,
  "title": "<title>"
}


plot_bar_chart:
Use for comparing one or more quantitative values across categorical or bucketed temporal groups.

Best for:
- "Y by A"
- "Y per A"
- "Y by A and B"
- totals, averages, counts, rankings, grouped comparisons

Mapping rules:
- For "Y by A": use x = A, y = Y, group_by = [A].
- For "Y by A and B": use x = A, color_by = B, y = Y, group_by = [A, B].
- For "Y by A, B, and C": use x = A, color_by = B or C, and group_by = [A, B, C].
- If two columns together define the x-axis label, create the combined column first with preprocess_for_chart and use it as x.
- group_by must include every column needed to preserve the requested breakdown.
- Use aggregate = "sum" by default for additive quantities unless the query specifies another aggregation.

Do not use a bar chart for multi-period time-series trends when a line chart can show the evolution more clearly.

A temporal column does not automatically make a bar chart appropriate.
Use bars for discrete period totals only when the user explicitly asks to compare independent periods or requests a bar chart.

Args:
{
  "x": "<category_or_bucket_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>"],
  "color_by": null | "<secondary_category_col>",
  "orientation": "v|h",
  "barmode": "group|stack|relative",
  "title": "<title>"
}

plot_line_chart:
Use for trends, time series, ordered progression, or cumulative values over time.

Use a line chart when:
- x is a date, year, month, quarter, or another ordered temporal column;
- the user asks for yearly, monthly, quarterly, or daily evolution;
- the chart shows how a metric changes across multiple time periods;
- multiple entities should be represented as separate time-series traces.

For "Y by time for each A":
- x = time column
- y = Y
- series_by = A
- each unique series_by value becomes one trace

Prefer a line chart over a bar chart whenever the main purpose is to show change or evolution over time.

Trace rules:
- Use series_by when one column defines separate traces.
- series_by values become the trace names.
- Use series_by = NAME when each well should be a separate trace.
- If y is a list and series_by is provided, traces are named "<series_by value> - <y column>".
- If series_by is null, traces are named from y column names.
- color_by is deprecated. Use series_by instead.

Args:
{
  "x": "<time_or_ordered_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>", "..."],
  "series_by": null | "<category_col>",
  "date_bucket": null | "D|W|M|Q|Y",
  "cumulative": true|false,
  "title": "<title>"
}

plot_pie_chart:
Use only for part-to-whole/share/composition questions.
Args:
{
  "labels": "<category_col>",
  "values": "<numeric_col>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<label_col>"],
  "hole": 0.0,
  "title": "<title>"
}

plot_scatter_chart:
Use for numeric-vs-numeric relationships, correlations, crossplots, clusters, or row-level comparisons.
Args:
{
  "x": "<numeric_col>",
  "y": "<numeric_col_or_list>",
  "series_by": null | "<category_col>",
  "size_by": null | "<numeric_col>",
  "text_by": null | "<label_col>",
  "title": "<title>"
}

IMPORTANT
YOU MUST address only the parts of the user question for which the table is related
YOU MUST Ignore the parts of the question that the information in the table cannot address
             

RULES
- Return only valid JSON.
- Do not invent tools.
- Do not invent arguments.
- Use only columns that exist or are created by preprocess_for_chart.
- Prefer no preprocess when existing columns are sufficient.
- Use sum by default for additive quantities unless otherwise specified.
- If uncertain, return {"reason": "...", "preprocess": null, "plot": null}.
- When multiple temporal dimensions together define the displayed x-axis grouping
(e.g. year + quarter, year + month),
create a combined temporal category for x.

OUTPUT SHAPE
{

  "preprocess": [],
  "plot": {
    "tool": "<plot_tool>",
    "args": {}
  }
}
"""

chart_agent_prompt2 = CHART_AGENT_PROMPTV4

c = PresenterConfig( )
c.prompt = chart_agent_prompt 

In [ ]:
query = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production since year 2015 for all the wells 
"""

planner = vis_system.planner_component
plan = planner.run( query )


In [ ]:


query2 = """Explain VRR briefly and then:

1. plot the yearly liquid production of the 5 top producers based on the cummulated oil production in 2018, 
2. show the cummulated liquid production since year 2015 for the first two of those wells.  

"""


#this is what the presenter consumes 
execution_state = vis_system.run( query2 )



In [ ]:
import pickle
with open("execution_state.pkl", "wb") as file:
    pickle.dump(execution_state, file)


In [ ]:
import pickle 
with open("execution_state.pkl", "rb") as file:
    loaded_data = pickle.load(file)

#import pickle
#with open("execution_state.pkl", "wb") as file:
#    pickle.dump(execution_state, file)

execution_state = loaded_data

In [ ]:
execution_state

In [ ]:

from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
print(ui_items)

for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 




In [ ]:

from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
ui_items

In [ ]:
ui_items.items

In [ ]:
for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 
        

# END 

In [ ]:
#item = ui_items.items[1]
#item = item.data['plotly']
#pio.show(item)

item = ui_items.items[3]
item = item.data['plotly']
pio.show(item)




# Improved the presenter. 

In [ ]:
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if

from visualization_system.visualization_backend.all_classes import * 



In [ ]:
llm = azure_llm_if()
import pickle 
with open("execution_state.pkl", "rb") as file:
    loaded_data = pickle.load(file)

#import pickle
#with open("execution_state.pkl", "wb") as file:
#    pickle.dump(execution_state, file)

execution_state = loaded_data


presenter = PresenterComponent4(llm)

presenter_response = presenter.run(execution_state)

ui_items = presenter_response.items

ui_items 

In [ ]:
for item in ui_items:
    if item.type=='text':
        print( item.data['text'])
    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


# END 

In [ ]:
from visualization_system.visualization_backend.analyst.prompts import chart_agent_prompt
from visualization_system.visualization_backend.analyst.prompts import small_table_prompt
from visualization_system.visualization_backend.analyst.prompts import split_subinstructions_prompt

class SubInstruction(BaseModel):
    """
    One presentation item to produce from one source.
    """

    sub_instruction: str = Field(
        description=(
            "The specific part of the original instruction that this source "
            "should answer."
        )
    )

    kind: Literal["table", "text"] = Field(
        description="Whether the source is a table or a text result."
    )

    source_id: str = Field(
        description=(
            "The exact SOURCE_ID provided in the available sources. "
            "For tables use the table name. "
            "For text use the text SOURCE_ID."
        )
    )


class SubInstructions(BaseModel):
    """
    Ordered presentation plan for one TaskResult.
    """

    items: list[SubInstruction] = Field(
        description=(
            "The ordered list of presentation items to generate."
        )
    )
    
    
class PresenterChartingTools:

    ALLOWED_AGGS = {"sum", "mean", "median", "min", "max", "count", "nunique"}


    plotly_config = {
                "responsive": True,
                "displaylogo": False,
            }



    def run_preprocess(
        self,
        df: pd.DataFrame,
        preprocess_steps: list[dict] | None,
    ) -> pd.DataFrame:
        work = df.copy()

        for step in preprocess_steps or []:
            
            operation = step.get("operation")

            if not operation:
                raise ValueError("Preprocess step is missing 'operation'")

            args = step.get("args") or {}

            work = self.run_preprocess_operation(
                operation=operation,
                df=work,
                args=args,
            )

        return work


    def run_preprocess_operation(
        self,
        operation: str,
        df: pd.DataFrame,
        args: dict[str, Any],
    ) -> pd.DataFrame:
        preprocess_tools = {
            "filter_rows": self.filter_rows,
            "aggregate": self.aggregate_for_chart,
            "sort_rows": self.sort_rows,
            "limit_rows": self.limit_rows,
            "select_columns": self.select_columns,
            "select_top_entities": self.select_top_entities,
            "create_combined_category": self.create_combined_category,
            "create_date_bucket": self.create_date_bucket,
        }

        if operation not in preprocess_tools:
            raise ValueError(f"Unknown preprocess operation: {operation}")

        return preprocess_tools[operation](
            df=df,
            **args,
        )


    ##########################
    #       pre-process      # 
    ##########################
    def filter_rows(self,df: pd.DataFrame,filters: list[dict]) -> pd.DataFrame:
        
        work = df.copy()
        for item in filters:
            column = item["column"]
            operator = item["operator"]
            value = item["value"]

            self._validate_columns(work, [column])

            if operator == "==":
                work = work[work[column] == value]
            elif operator == "!=":
                work = work[work[column] != value]
            elif operator == ">":
                work = work[work[column] > value]
            elif operator == ">=":
                work = work[work[column] >= value]
            elif operator == "<":
                work = work[work[column] < value]
            elif operator == "<=":
                work = work[work[column] <= value]
 


            elif operator == "in":
                if not isinstance(value, (list, tuple, set)):
                    raise ValueError("'in' filter value must be a list")
                work = work[work[column].isin(value)]

            elif operator == "not_in":
                if not isinstance(value, (list, tuple, set)):
                    raise ValueError("'not_in' filter value must be a list")
                work = work[~work[column].isin(value)]




            else:
                raise ValueError(f"Unsupported filter operator: {operator}")

        return work

    def aggregate_for_chart( self, df: pd.DataFrame, group_by: list[str],
        metrics: dict[str, str],
    ) -> pd.DataFrame:
        
        self._validate_columns(df, group_by)

        for column, aggregate in metrics.items():
            self._validate_columns(df, [column])

            if aggregate not in self.ALLOWED_AGGS:
                raise ValueError(f"Unsupported aggregate: {aggregate}")

        return (df.groupby(group_by, dropna=False, as_index=False).agg(metrics))

    def sort_rows(
        self,
        df: pd.DataFrame,
        sort_by: str | list[str],
        ascending: bool = True,
    ) -> pd.DataFrame:
        sort_columns = self._as_list(sort_by)
        self._validate_columns(df, sort_columns)

        return df.sort_values(
            sort_columns,
            ascending=ascending,
        )

    def limit_rows(
        self,
        df: pd.DataFrame,
        n: int,
    ) -> pd.DataFrame:
        return df.head(n)

    def select_columns(
        self,
        df: pd.DataFrame,
        columns: list[str],
    ) -> pd.DataFrame:
        self._validate_columns(df, columns)
        return df[columns].copy()

    def create_combined_category(
        self,
        df: pd.DataFrame,
        col1: str,
        col2: str,
        new_col: str | None = None,
        sep: str = " / ",
    ) -> pd.DataFrame:
        out = df.copy()

        self._validate_columns(out, [col1, col2])

        new_col = new_col or f"{col1}_{col2}"

        out[new_col] = (
            out[col1].fillna("").astype(str)
            + sep
            + out[col2].fillna("").astype(str)
        )

        return out

    def create_date_bucket(
        self,
        df: pd.DataFrame,
        date_col: str,
        bucket: str,
        new_col: str | None = None,
    ) -> pd.DataFrame:
        out, generated_col = self._bucket_date(
            df,
            date_col,
            bucket,
        )

        if new_col and new_col != generated_col:
            out = out.rename(
                columns={generated_col: new_col}
            )

        return out

    def select_top_entities(
        self,
        df: pd.DataFrame,
        entity_col: str,
        metric_col: str,
        n: int,
        aggregate: str = "sum",
        ascending: bool = False,
        filters: list[dict] | None = None,
        keep_all_rows: bool = True,
    ) -> pd.DataFrame:
        
        self._validate_columns(df,[entity_col, metric_col])
        if aggregate not in self.ALLOWED_AGGS:
            raise ValueError(f"Unsupported aggregate: {aggregate}")

        if n <= 0:
            raise ValueError("n must be greater than zero")
        
        ranking_data = df.copy()

        if filters:
            ranking_data = self.filter_rows(
                ranking_data,
                filters,
            )

        ranking = (
            ranking_data
            .groupby(entity_col, dropna=False, as_index=False)[metric_col]
            .agg(aggregate)
            .sort_values(metric_col, ascending=ascending)
            .head(n)
        )

        selected_entities = ranking[entity_col].tolist()

        if keep_all_rows:
            return df[df[entity_col].isin(selected_entities)].copy()

        return ranking



    def run_plot_tool(
        self,
        tool_name: str,
        df: pd.DataFrame,
        args: dict[str, Any],
    ) -> dict:
        plotting_tools = {
            "plot_bar_chart": self.plot_bar_chart,
            "plot_line_chart": self.plot_line_chart,
            "plot_pie_chart": self.plot_pie_chart,
            "plot_scatter_chart": self.plot_scatter_chart,
            "plot_list": self.plot_list,
        }

        if tool_name not in plotting_tools:
            raise ValueError(f"Unknown plot tool: {tool_name}")

        args = self._filter_args(tool_name, args)
        return plotting_tools[tool_name](df=df, **args)

    def format_label(self, name: str) -> str:
        """
        Convert column-like names to display labels.

        Examples:
        - year_quarter -> Year quarter
        - percentage_contribution -> Percentage contribution
        - TOTAL_WATER_INJECTION_VOLUME -> Total water injection volume
        """
        if name is None:
            return ""

        text = str(name).replace("_", " ").strip().lower()
        return text[:1].upper() + text[1:]

    def _as_list(self, value):
        if value is None:
            return []
        return [value] if isinstance(value, str) else list(value)

    def _strip_markdown_json(self, text: str) -> str:
        """
        Remove markdown code fences from LLM JSON responses.

        Examples:
        ```json
        {...}
        ```

        ->
        {...}
        """

        text = text.strip()

        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

        return text.strip()

    def _validate_columns(
        self,
        df: pd.DataFrame,
        columns: list[str],
        label: str = "column",
    ):
        """
        Validate that all requested columns exist in the dataframe.
        """

        missing = [c for c in columns if c not in df.columns]

        if missing:
            raise ValueError(f"Missing {label}(s): {missing}")

    def _filter_args(
        self,
        tool_name: str,
        args: dict[str, Any],
    ) -> dict[str, Any]:
        """
        Remove unsupported arguments generated by the LLM.
        """

        allowed_args = {
            "plot_bar_chart": {
                "x",
                "y",
                "aggregate",
                "group_by",
                "color_by",
                "orientation",
                "barmode",
                "title",
                "template",
            },
            "plot_line_chart": {
                "x",
                "y",
                "aggregate",
                "group_by","series_by",
                "color_by",
                "date_bucket",
                "cumulative",
                "title",
                "template",
            },
            "plot_pie_chart": {
                "labels",
                "values",
                "aggregate",
                "group_by",
                "title",
                "hole",
                "template",
            },
            "plot_scatter_chart": {
                "x",
                "y",
                "color_by",
                "size_by",
                "text_by",
                "title",
                "template",
            },
            "plot_list": {
                "columns",
                "sort_by",
                "sort_order",
                "limit",
                "title",
            },
        }

        if tool_name not in allowed_args:
            raise ValueError(f"Unknown tool: {tool_name}")

        return {
            k: v
            for k, v in args.items()
            if k in allowed_args[tool_name]
        }

    def _aggregate(
        self,
        df: pd.DataFrame,
        group_by: list[str],
        value_cols: list[str],
        aggregate: str,
    ) -> pd.DataFrame:
        if aggregate not in self.ALLOWED_AGGS:
            raise ValueError(f"Unsupported aggregate: {aggregate}")

        self._validate_columns(df, group_by, "group_by column")
        self._validate_columns(df, value_cols, "value column")

        return (
            df.groupby(group_by, dropna=False, as_index=False)[value_cols]
            .agg(aggregate)
        )

    def _bucket_date(
        self,
        df: pd.DataFrame,
        date_col: str,
        bucket: str,
    ) -> tuple[pd.DataFrame, str]:
        """
        Create a date bucket column.

        bucket:
        - "D": day
        - "W": week
        - "M": month
        - "Q": quarter
        - "Y": year
        """

        out = df.copy()
        bucket_col = f"{date_col}_{bucket}"

        self._validate_columns(out, [date_col])

        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

        if bucket == "D":
            out[bucket_col] = out[date_col].dt.to_period("D").dt.to_timestamp()
        elif bucket == "W":
            out[bucket_col] = out[date_col].dt.to_period("W").dt.start_time
        elif bucket == "M":
            out[bucket_col] = out[date_col].dt.to_period("M").dt.to_timestamp()
        elif bucket == "Q":
            out[bucket_col] = out[date_col].dt.to_period("Q").dt.to_timestamp()
        elif bucket == "Y":
            out[bucket_col] = out[date_col].dt.to_period("Y").dt.to_timestamp()
        else:
            raise ValueError("date_bucket must be one of: D, W, M, Q, Y")

        return out, bucket_col

    def plot_bar_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        color_by: str | None = None,
        orientation: str = "v",
        barmode: str = "group",
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        required = [x, *y_cols]
        if color_by:
            required.append(color_by)

        self._validate_columns(df, required)

        work = df.copy()

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [x]

            if x not in group_cols:
                group_cols.insert(0, x)

            if color_by and color_by not in group_cols:
                group_cols.append(color_by)

            work = self._aggregate(work, group_cols, y_cols, aggregate)

        data = []
        groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

        for group_value, g in groups:
            for y_col in y_cols:
                if group_value is None:
                    name = self.format_label(y_col)
                else:
                    name = self.format_label(str(group_value))

                if group_value is not None and len(y_cols) > 1:
                    name = f"{self.format_label(str(group_value))} - {self.format_label(y_col)}"

                trace = {
                    "type": "bar",
                    "name": name,
                    "orientation": orientation,
                }

                if orientation == "h":
                    trace["x"] = g[y_col].tolist()
                    trace["y"] = g[x].astype(str).tolist()
                else:
                    trace["x"] = g[x].astype(str).tolist()
                    trace["y"] = g[y_col].tolist()

                data.append(trace)

        y_label = self.format_label(", ".join(y_cols))
        x_label = self.format_label(x)

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": title or f"{y_label} by {x_label}"
                },
                "xaxis": {
                    "title": {
                        "text": y_label if orientation == "h" else x_label
                    }
                },
                "yaxis": {
                    "title": {
                        "text": x_label if orientation == "h" else y_label
                    }
                },
                "barmode": barmode,
                # "template": template,
            },
            "config": {
                "responsive": True,
                "displaylogo": False,
            },
        }

    def plot_line_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        series_by: str | None = None,
        color_by: str | None = None,  # backwards compatibility
        date_bucket: str | None = None,
        cumulative: bool = False,
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        if series_by is None:
            series_by = color_by

        required = [x, *y_cols]
        if series_by:
            required.append(series_by)

        self._validate_columns(df, required)

        work = df.copy()
        x_plot = x

        if date_bucket is not None:
            work, x_plot = self._bucket_date(work, x, date_bucket)

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [x_plot]

            if x_plot not in group_cols:
                group_cols.insert(0, x_plot)

            if series_by and series_by not in group_cols:
                group_cols.append(series_by)

            work = self._aggregate(work, group_cols, y_cols, aggregate)

        sort_cols = [series_by, x_plot] if series_by else [x_plot]
        work = work.sort_values(sort_cols)

        if cumulative:
            if series_by:
                for col in y_cols:
                    work[col] = work.groupby(series_by, dropna=False)[col].cumsum()
            else:
                for col in y_cols:
                    work[col] = work[col].cumsum()

        data = []
        groups = work.groupby(series_by, dropna=False) if series_by else [(None, work)]

        total_points = len(work) * len(y_cols)
        disable_all_markers = total_points > 2000

        for group_value, g in groups:
            for y_col in y_cols:
                if group_value is None:
                    name = self.format_label(y_col)
                elif len(y_cols) == 1:
                    name = str(group_value)
                else:
                    name = f"{group_value} - {self.format_label(y_col)}"

                data.append({
                    "type": "scatter",
                    "mode": "lines",
                    "x": g[x_plot].tolist(),
                    "y": g[y_col].tolist(),
                    "name": name,
                })

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": self.format_label(title) or f"{', '.join(y_cols)} over {x}"
                },
                "xaxis": {
                    "title": {
                        "text": self.format_label(x)
                    }
                },
                "yaxis": {
                    "title": {
                        "text": self.format_label(", ".join(y_cols))
                    }
                },
            },
            "config": self.plotly_config,
        }

    def plot_pie_chart(
        self,
        df: pd.DataFrame,
        labels: str,
        values: str,
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        title: str | None = None,
        hole: float = 0.0,
        # template: str = "plotly_white",
    ) -> dict:
        self._validate_columns(df, [labels, values])

        work = df.copy()

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [labels]

            if labels not in group_cols:
                group_cols.insert(0, labels)

            work = self._aggregate(work, group_cols, [values], aggregate)

        return {
            "data": [
                {
                    "type": "pie",
                    "labels": work[labels].astype(str).tolist(),
                    "values": work[values].tolist(),
                    "hole": hole,
                }
            ],
            "layout": {
                "title": {
                    "text": title or f"{values} share by {labels}"
                },
                # "template": template,
            },
            "config": self.plotly_config
        }

    def plot_scatter_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        color_by: str | None = None,
        size_by: str | None = None,
        text_by: str | None = None,
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        required = [x, *y_cols]
        if color_by:
            required.append(color_by)
        if size_by:
            required.append(size_by)
        if text_by:
            required.append(text_by)

        self._validate_columns(df, required)

        work = df.copy()
        data = []
        groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

        total_points = len(work) * len(y_cols)
        disable_all_markers = total_points > 2000

        for group_value, g in groups:
            for y_col in y_cols:
                name = y_col if group_value is None else str(group_value)

                if group_value is not None and len(y_cols) > 1:
                    name = f"{group_value} - {y_col}"

                n_points = len(g)

                use_markers = (
                    not disable_all_markers
                    and n_points <= 100
                )

                mode = "markers" if use_markers else "lines"

                trace = {
                    "type": "scattergl",
                    "mode": mode,
                    "x": g[x].tolist(),
                    "y": g[y_col].tolist(),
                    "name": name,
                }

                if size_by:
                    size_values = pd.to_numeric(g[size_by], errors="coerce").fillna(0)
                    max_size = max(float(size_values.max()), 1.0)

                    trace["marker"] = {
                        "size": size_values.tolist(),
                        "sizemode": "area",
                        "sizeref": max_size / 40,
                        "sizemin": 4,
                    }

                if text_by:
                    trace["text"] = g[text_by].astype(str).tolist()
                    trace["hovertemplate"] = (
                        f"{x}: %{{x}}<br>"
                        f"{y_col}: %{{y}}<br>"
                        f"{text_by}: %{{text}}"
                        "<extra></extra>"
                    )

                data.append(trace)

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": title or f"{', '.join(y_cols)} vs {x}"
                },
                "xaxis": {
                    "title": {
                        "text": x
                    }
                },
                "yaxis": {
                    "title": {
                        "text": ", ".join(y_cols)
                    }
                },
                # "template": template,
            },
            "config": self.plotly_config
        }

    def plot_list(
        self,
        df: pd.DataFrame,
        columns: list[str] | str | None = None,
        *,
        sort_by: str | None = None,
        sort_order: str = "desc",
        limit: int | None = None,
        title: str | None = None,
    ) -> dict:
        work = df.copy()

        if columns is not None:
            columns = self._as_list(columns)
            self._validate_columns(work, columns)
            work = work[columns]

        if sort_by is not None:
            self._validate_columns(work, [sort_by])

            ascending = sort_order.lower() == "asc"
            work = work.sort_values(sort_by, ascending=ascending)

        if limit is not None:
            work = work.head(limit)

        header_values = [self.format_label(c) for c in work.columns]

        cell_values = []
        for col in work.columns:
            s = work[col]

            if pd.api.types.is_datetime64_any_dtype(s):
                values = s.dt.strftime("%Y-%m-%d").fillna("").tolist()
            else:
                values = s.fillna("").astype(str).tolist()

            cell_values.append(values)

        return {
            "data": [
                {
                    "type": "table",
                    "header": {
                        "values": header_values,
                        "align": "left",
                    },
                    "cells": {
                        "values": cell_values,
                        "align": "left",
                    },
                }
            ],
            "layout": {
                "title": {
                    "text": self.format_label(title) or "Table"
                },
            },
            "config": self.plotly_config
        }
    

class PresenterConfig:

    prompt :str = chart_agent_prompt
    split_subinstructions_prompr: str = split_subinstructions_prompt 
    small_table_prompt: str = small_table_prompt 
    
    def __init__(
        self,
        charting_tools: PresenterChartingTools | None = None,
    ):
        self.charting_tools = charting_tools or PresenterChartingTools()


class PresenterComponent4:
    """
    Converts an ExecutorState into UI display items.

    Each TaskResult is processed as a whole:
    - all TextResult and DataFrameResult objects are added to one context;
    - one LLM call splits the task instruction into sub-instructions;
    - each sub-instruction is associated with one source;
    - text sources become text UIItems;
    - table sources are passed to the existing dataframe presentation logic.
    """

    def __init__(
        self,
        llm: Any,
        config: PresenterConfig | None = None,
    ):
        self.llm = llm
        self.config = config or PresenterConfig()
        self.charting_tools = self.config.charting_tools

    def run(self, result_state: ExecutorState) -> PresenterResponse:
        return self.process_task_results(result_state)

    def _make_clarification_item(
        self,
        clarification_request: str,
    ) -> UIItem:
        return UIItem(
            id=f"question_{uuid4().hex[:8]}",
            type="question",
            title="Additional information required",
            data={"question": clarification_request},
        )

    def _make_text_item(
        self,
        data_result: TextResult,
    ) -> UIItem:
        return UIItem(
            id=f"text_{uuid4().hex[:8]}",
            type="text",
            title=None,
            data={"text": data_result.text},
        )

    def _make_error_item(
        self,
        task_result: TaskResult,
        data_result: object | None = None,
    ) -> UIItem:
        return UIItem(
            id=f"error_{uuid4().hex[:8]}",
            type="error",
            title="Presentation error",
            data={
                "message": f"No presenter for result from {task_result.agent}",
                "details": (
                    str(type(data_result))
                    if data_result is not None
                    else task_result.instruction
                ),
            },
        )

    def build_task_result_context(
        self,
        task_result: TaskResult,
    ) -> tuple[str, dict[str, TextResult | DataFrameResult]]:
        """
        Build:
        - one text context containing all TextResult and DataFrameResult objects;
        - a source map used later to recover the original result objects.
        """
        processor = TableResponseProcessor()

        context_parts: list[str] = []
        source_map: dict[str, TextResult | DataFrameResult] = {}

        for n, data_result in enumerate(task_result.data_results):

            if isinstance(data_result, TextResult):
                source_id = f"text_{n}"

                context_parts.append(
                    "\n".join([
                        f"SOURCE_ID: {source_id}",
                        "SOURCE_TYPE: text",
                        "CONTENT:",
                        data_result.text,
                    ])
                )

                source_map[source_id] = data_result

            elif isinstance(data_result, DataFrameResult):
                source_id = data_result.table_name
                table_context = processor.extract_table_context(data_result)

                context_parts.append(
                    "\n".join([
                        f"SOURCE_ID: {source_id}",
                        "SOURCE_TYPE: table",
                        table_context,
                    ])
                )

                source_map[source_id] = data_result

        context_text = "\n\n---\n\n".join(context_parts)

        return context_text, source_map

    def get_subinstructions(
        self,
        task_result: TaskResult,
        context_text: str,
    ) -> SubInstructions:
        """
        Split the task instruction and associate each sub-instruction
        with one available source.
        """
        messages = [
            SystemMessage(content=self.config.split_subinstructions_prompr),
            HumanMessage(
                content=(
                    f"INSTRUCTION\n"
                    f"{task_result.instruction}\n\n"
                    f"AVAILABLE SOURCES\n"
                    f"{context_text}"
                )
            ),
        ]

        structured_llm = self.llm.with_structured_output(SubInstructions)

        return structured_llm.invoke(messages)

    def process_single_task_result(
        self,
        task_result: TaskResult,
    ) -> list[UIItem]:
        context_text, source_map = self.build_task_result_context(
            task_result
        )

        sub_instructions = self.get_subinstructions(
            task_result=task_result,
            context_text=context_text,
        )

        ui_items: list[UIItem] = []

        for item in sub_instructions.items:
            source = source_map[item.source_id]

            if item.kind == "text":
                ui_items.append(
                    self._make_text_item(source)
                )

            elif item.kind == "table":
                ui_items.append(
                    self._process_dataframe(
                        source,
                        item.sub_instruction,
                    )
                )

        return ui_items

    def process_task_results(
        self,
        execution_state: ExecutorState,
    ) -> PresenterResponse:
        ui_items: list[UIItem] = []

        clarification_request = execution_state.get(
            "clarification_request"
        )

        if clarification_request:
            ui_items.append(
                self._make_clarification_item(
                    clarification_request
                )
            )

            return PresenterResponse(items=ui_items)

        for task_result in execution_state.get("task_results", []):
            ui_task_items = self.process_single_task_result(
                task_result
            )

            ui_items.extend(ui_task_items)

        return PresenterResponse(items=ui_items)

    def _present_very_small_table(
        self,
        df: pd.DataFrame,
        data_result: DataFrameResult,
        instruction: str,
    ) -> UIItem:
        data_string = df.to_json()

        print('processing very small table')
        prompt = (
            self.config.small_table_prompt
            + "\n\n"
            + (
                "### Context\n"
                f"- Table Name: {getattr(data_result, 'table_name', 'N/A')}\n"
                f"- Description: "
                f"{getattr(data_result, 'description', 'No description provided.')}\n\n"
                "### Data\n"
                f"{data_string}\n\n"
                "User question:\n"
                f"{instruction}\n"
            )
        )

        response = self.llm.invoke(prompt)

        text_output = (
            response.content
            if hasattr(response, "content")
            else str(response)
        )

        return UIItem(
            id=f"text_{uuid4().hex[:8]}",
            type="text",
            title=None,
            data={"text": text_output},
        )

    def _make_chart_item(
        self,
        figure_title: str,
        plotly_json_figure: dict,
        description: str | None = None,
    ) -> UIItem:
        return UIItem(
            id=f"chart_{uuid4().hex[:8]}",
            type="chart",
            title=figure_title,
            data={
                "engine": "plotly",
                "plotly": plotly_json_figure,
            },
            meta={
                "description": description,
            },
        )


    def _run_chart_plan(
        self,
        plan: dict,
        df: pd.DataFrame,
    ) -> dict | None:
        work = df.copy()

        preprocess_steps = plan.get("preprocess") or []
        plot = plan.get("plot")

        if preprocess_steps:
            work = self.charting_tools.run_preprocess(
                work,
                preprocess_steps,
            )

        if not plot:
            return None

        tool_name = plot.get("tool")
        args = plot.get("args") or {}

        return self.charting_tools.run_plot_tool(
            tool_name=tool_name,
            df=work,
            args=args,
        )


    def _format_label(
        self,
        name: str,
    ) -> str:
        if name is None:
            return ""

        text = str(name).replace("_", " ").strip().lower()

        return text[:1].upper() + text[1:]

    def _process_dataframe(
        self,
        data_result: DataFrameResult,
        instruction: str,
    ) -> UIItem:
        df = data_result.dataframe
        nrows, ncols = df.shape

        if nrows <= 2 and ncols <= 2:
            return self._present_very_small_table(
                df,
                data_result,
                instruction,
            )

        processor = TableResponseProcessor()

        table_context = processor.extract_table_context(
            data_result
        )

        chart_plan = self._select_chart_plan(
            instruction,
            table_context,
        )

        print('****chart plan****')
        print(instruction)
        print(chart_plan)


        chart_output = self._run_chart_plan(
            chart_plan,
            df,
        )

        return self._make_chart_item(
            self._format_label(data_result.table_name),
            chart_output,
            data_result.description,
        )

    def _strip_markdown_json(
        self,
        text: str,
    ) -> str:
        text = text.strip()

        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
        )

        text = re.sub(
            r"\s*```$",
            "",
            text,
        )

        return text.strip()

    def _select_chart_plan(
        self,
        user_query: str,
        table_context: str,
    ) -> dict:
        messages = [
            SystemMessage(content=self.config.prompt),
            HumanMessage(
                content=(
                    f"USER QUERY\n"
                    f"{user_query}\n\n"
                    f"TABLE\n"
                    f"{table_context}"
                )
            ),
        ]

        response = self.llm.invoke(messages)

        text = self._strip_markdown_json(
            response.content
        )

        return json.loads(text)
    

    

In [ ]:
#pprint.pprint( execution_state['task_results'][1] )
t = execution_state['task_results'][1]
instruction = t.instruction 
#t.data_results.insert(0, TextResult(text="All the fruits in the basket are sweet."))

print( instruction )
p = TableResponseProcessor()
context = [] 

aux= {} 
for n,data_result in enumerate(t.data_results):


    if isinstance( data_result, TextResult):
        print("processing text ")
        context.append( data_result.text )
        aux[n] = data_result.text 

    if isinstance(data_result, DataFrameResult):
        print("processing dataframe result")
        table_context = p.extract_table_context( data_result )
        context.append( table_context )
        aux[ data_result.table_name ] = (table_context,data_result)
    
prompt = """You will receive an 'instruction' and information of tables and text. your job is to 
analyze the instruction. It might contain several sub-instructions. 
Decide what parts of the information available can be used to execute the 
instruction and its sub-instructions. 

Do not explain anything, do not add more details than strictly needed to produce the required output 
"""
class SubInstruction(BaseModel):
    sub_instruction: str = Field(
        description="The specific part of the instruction."
    )
    kind:  Literal['table','text']
    
    data: str  = Field(
        description="The name of the table or the textual information "
    )
  
  
class SubInstructions(BaseModel):

    items: List[SubInstruction] =  Field(description="The list of sub-instructions and the information relevant for each")

context = "\n\n".join(context)

#instruction2 = "Check if the fruits are sour or sweet, " + instruction# then list the top 5 producers by cummulated oil produced in 2018 "

context =  "\n" + context + "\n\n" + instruction + "\n\n"
messages = [SystemMessage(prompt + "\n\n" + context )]

structured_llm = llm.with_structured_output(SubInstructions)
response = structured_llm.invoke(messages)

In [ ]:
pprint.pprint(context)


In [ ]:
pprint.pprint(response.items)


In [ ]:

items = [] 

for i in response.items:
    if i.kind=='text':
        print("It is a text")
        r =  UIItem(
                id=f"text_{uuid4().hex[:8]}",
                type="text",
                title=None,
                data={"text": i.data}
            )
        items.append( r )

    if i.kind=='table':

        #print( '**',i.sub_instruction,'**')
        table_name = i.data 
        acontext = aux[table_name ][0]
        data_result = aux[table_name][1]
        #print('table', table_name, acontext )
        
        r = presenter._process_dataframe(data_result,i.sub_instruction)
        items.append( r )

        


In [ ]:
print( items )

print()
print()
print()

for item in items:

    if item.type=='text':
        print( item.data['text'])

    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


## More organized

In [ ]:
from typing import Literal
from uuid import uuid4

from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage


# =============================================================================
# Structured output models
# =============================================================================

class SubInstruction(BaseModel):
    sub_instruction: str = Field(
        description=(
            "The specific part of the original instruction that must be "
            "presented using the selected source."
        )
    )

    kind: Literal["table", "text"] = Field(
        description="The type of source associated with this sub-instruction."
    )

    source_id: str = Field(
        description=(
            "The exact SOURCE ID provided in the available sources. "
            "For a table, this is the exact table name. "
            "For text, this is the exact text result identifier."
        )
    )


class SubInstructions(BaseModel):
    items: list[SubInstruction] = Field(
        description=(
            "The requested outputs, in the order in which they should be "
            "presented. Irrelevant and intermediate sources must be omitted."
        )
    )


# =============================================================================
# Routing prompt
# =============================================================================

TASK_PRESENTATION_ROUTER_PROMPT = """
You receive one instruction and a set of available sources.

The instruction may contain several sub-instructions.

Your job is to:

1. Identify each distinct output explicitly requested by the instruction.
2. Match each requested output to exactly one relevant source.
3. Return the outputs in the order in which they should be presented.
4. Use the exact SOURCE ID provided for each source.
5. Omit sources that are irrelevant or only intermediate calculation results.
6. Do not invent facts, tables, source IDs, calculations, or additional requests.
7. Do not explain your decisions.
8. Do not create an item when the available sources cannot support it.
9. Do not repeat the same source unless it is genuinely required for two
   different requested outputs.

For a text source:
- Use it when the source directly contains the requested textual answer.
- The sub_instruction should describe the part of the instruction answered
  by the text.
- The source_id must be the exact text SOURCE ID.

For a table source:
- Use it when the table contains the information required for the requested
  table, chart, list, ranking, comparison, or numerical presentation.
- The sub_instruction must contain only the part of the original instruction
  that the selected table can address.
- The source_id must be the exact table SOURCE ID.

Important:
- A table used only to calculate another final table is usually an intermediate
  source and should be omitted unless the user explicitly requested it.
- Do not return the source content itself.
- Return only the structured result.
"""


# =============================================================================
# Select the TaskResult to process
# =============================================================================

task_result_index = 1

task_result = execution_state["task_results"][task_result_index]
instruction = task_result.instruction

print("TASK INSTRUCTION")
print(instruction)
print()


# =============================================================================
# Build source context and source lookup
# =============================================================================

table_processor = TableResponseProcessor()

source_contexts: list[str] = []
source_lookup: dict[str, TextResult | DataFrameResult] = {}

for result_index, data_result in enumerate(task_result.data_results):

    if isinstance(data_result, TextResult):
        source_id = f"text_result_{result_index}"

        source_contexts.append(
            "\n".join([
                f"SOURCE ID: {source_id}",
                "SOURCE KIND: text",
                "CONTENT:",
                data_result.text,
            ])
        )

        source_lookup[source_id] = data_result

    elif isinstance(data_result, DataFrameResult):
        source_id = data_result.table_name
        table_context = table_processor.extract_table_context(data_result)

        source_contexts.append(
            "\n".join([
                f"SOURCE ID: {source_id}",
                "SOURCE KIND: table",
                table_context,
            ])
        )

        source_lookup[source_id] = data_result

    else:
        print(
            "Ignoring unsupported data result:",
            type(data_result).__name__,
        )


available_sources_context = "\n\n---\n\n".join(source_contexts)

print("AVAILABLE SOURCE IDS")
for source_id, source in source_lookup.items():
    print(f"- {source_id}: {type(source).__name__}")
print()


# =============================================================================
# One LLM call to split the instruction and route each part to a source
# =============================================================================

messages = [
    SystemMessage(content=TASK_PRESENTATION_ROUTER_PROMPT),
    HumanMessage(
        content=(
            f"ORIGINAL INSTRUCTION\n"
            f"{instruction}\n\n"
            f"AVAILABLE SOURCES\n"
            f"{available_sources_context}"
        )
    ),
]

structured_llm = llm.with_structured_output(SubInstructions)
routing_response = structured_llm.invoke(messages)

print("ROUTING RESPONSE")
for routed_item in routing_response.items:
    print(routed_item)
print()


# =============================================================================
# Convert the routed outputs into UIItems
# =============================================================================

items: list[UIItem] = []

for routed_item in routing_response.items:
    source = source_lookup.get(routed_item.source_id)

    if source is None:
        items.append(
            UIItem(
                id=f"error_{uuid4().hex[:8]}",
                type="error",
                title="Presentation error",
                data={
                    "message": (
                        "The presentation router selected an unknown source."
                    ),
                    "details": routed_item.source_id,
                },
            )
        )
        continue

    if routed_item.kind == "text":
        if not isinstance(source, TextResult):
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            "The presentation router classified a non-text "
                            "source as text."
                        ),
                        "details": routed_item.source_id,
                    },
                )
            )
            continue

        items.append(
            UIItem(
                id=f"text_{uuid4().hex[:8]}",
                type="text",
                title=None,
                data={
                    "text": source.text,
                },
                meta={
                    "sub_instruction": routed_item.sub_instruction,
                    "source_id": routed_item.source_id,
                },
            )
        )

    elif routed_item.kind == "table":
        if not isinstance(source, DataFrameResult):
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            "The presentation router classified a non-table "
                            "source as a table."
                        ),
                        "details": routed_item.source_id,
                    },
                )
            )
            continue

        try:
            ui_item = presenter._process_dataframe(
                source,
                routed_item.sub_instruction,
            )

            if ui_item is not None:
                ui_item.meta = {
                    **ui_item.meta,
                    "sub_instruction": routed_item.sub_instruction,
                    "source_id": routed_item.source_id,
                }
                items.append(ui_item)

        except Exception as exc:
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            f"Could not present table "
                            f"{routed_item.source_id}."
                        ),
                        "details": str(exc),
                    },
                    meta={
                        "sub_instruction": routed_item.sub_instruction,
                        "source_id": routed_item.source_id,
                    },
                )
            )


# =============================================================================
# Final presenter response
# =============================================================================

presenter_response = PresenterResponse(items=items)

print("GENERATED UI ITEMS")
for item in presenter_response.items:
    print(
        {
            "id": item.id,
            "type": item.type,
            "title": item.title,
            "source_id": item.meta.get("source_id"),
            "sub_instruction": item.meta.get("sub_instruction"),
        }
    )

presenter_response
  

In [ ]:
items = presenter_response.items 
print( items )

print()
print()
print()

for item in items:

    if item.type=='text':
        print( item.data['text'])

    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


In [ ]:


presenter = PresenterComponent4( llm )
#presenter = PresenterComponent2( llm )

ui_items = presenter.run( execution_state )
ui_items

In [ ]:
ui_items.items

In [ ]:
item = ui_items.items[1]
item = item.data['plotly']
pio.show(item)

item = ui_items.items[2]
item = item.data['plotly']
pio.show(item)

item = ui_items.items[3]
item = item.data['plotly']
pio.show(item)